## config.py

In [ ]:
"""
Script de test complet pour l'API RAG - Normes Électriques
Teste tous les endpoints disponibles (Version sans dépendances externes)
"""

import requests
import json
from typing import Dict, Any
import time

# Configuration
BASE_URL = " http://0.0.0.0:8000"
API_PREFIX = "/api/v1"


class RAGAPITester:
    """Classe pour tester l'API RAG"""
    
    def __init__(self, base_url: str = BASE_URL):
        self.base_url = base_url
        self.session = requests.Session()
        self.results = []
    
    def print_header(self, text: str):
        """Affiche un header"""
        print(f"\n{'='*80}")
        print(f"{text.center(80)}")
        print(f"{'='*80}")
    
    def print_success(self, text: str):
        """Affiche un message de succès"""
        print(f"✅ {text}")
    
    def print_error(self, text: str, detail: str = None):
        """Affiche un message d'erreur"""
        print(f"❌ {text}")
        if detail:
            print(f"   Détails: {detail}")
    
    def print_info(self, text: str):
        """Affiche un message d'information"""
        print(f"ℹ️  {text}")
    
    def test_health(self) -> bool:
        """Test du endpoint /health"""
        self.print_header("TEST 1: Health Check")
        try:
            response = self.session.get(f"{self.base_url}/health")
            response.raise_for_status()
            data = response.json()
            
            self.print_success(f"Status: {response.status_code}")
            print(f"Réponse: {json.dumps(data, indent=2, ensure_ascii=False)}")
            
            self.results.append(("Health Check", "✅ Réussi"))
            return True
        except Exception as e:
            self.print_error(f"Erreur: {e}")
            self.results.append(("Health Check", f"❌ Échoué: {e}"))
            return False
    
    def test_status(self) -> bool:
        """Test du endpoint /api/v1/status"""
        self.print_header("TEST 2: Status des Services")
        try:
            response = self.session.get(f"{self.base_url}{API_PREFIX}/status")
            response.raise_for_status()
            data = response.json()
            
            self.print_success(f"Status: {response.status_code}")
            print(f"Réponse: {json.dumps(data, indent=2, ensure_ascii=False)}")
            
            self.results.append(("Status Services", "✅ Réussi"))
            return True
        except Exception as e:
            self.print_error(f"Erreur: {e}")
            self.results.append(("Status Services", f"❌ Échoué: {e}"))
            return False
    
    def test_autocomplete(self) -> bool:
        """Test du endpoint /api/v1/autocomplete"""
        self.print_header("TEST 3: Autocomplétion")
        
        test_cases = [
            {"query": "protéger les", "max_results": 10},
            {"query": "remplacer le câble", "max_results": 10},
            {"query": "utiliser des", "max_results": 10},
        ]
        
        success_count = 0
        for i, test_data in enumerate(test_cases, 1):
            self.print_info(f"Test 3.{i}: query='{test_data['query']}'")
            try:
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/autocomplete",
                    json=test_data
                )
                response.raise_for_status()
                data = response.json()
                
                suggestions = data.get('suggestions', [])
                self.print_success(f"Suggestions reçues: {len(suggestions)}")
                
                # Afficher les premières suggestions
                if suggestions:
                    print(f"  Exemples de suggestions:")
                    for j, sugg in enumerate(suggestions[:3], 1):
                        print(f"    {j}. {sugg}")
                
                success_count += 1
            except requests.exceptions.HTTPError as e:
                error_detail = e.response.json() if e.response.text else str(e)
                self.print_error(f"Erreur HTTP {e.response.status_code}", json.dumps(error_detail, indent=2, ensure_ascii=False))
            except Exception as e:
                self.print_error(f"Erreur: {e}")
        
        result = f"✅ {success_count}/{len(test_cases)} réussis" if success_count == len(test_cases) else f"⚠️  {success_count}/{len(test_cases)} réussis"
        self.results.append(("Autocomplétion", result))
        return success_count == len(test_cases)
    
    def test_extract_norme(self) -> bool:
        """Test du endpoint /api/v1/extract_norme"""
        self.print_header("TEST 4: Extraction de Norme")
        
        test_cases = [
            "Protéger les circuits par un disjoncteur différentiel 30mA",
            "Remplacer le câble par un conducteur de section 2.5mm²",
            "Installer un tableau conforme à la NF C 15-100",
        ]
        
        success_count = 0
        for i, text in enumerate(test_cases, 1):
            self.print_info(f"Test 4.{i}: '{text}'")
            try:
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/extract_norme",
                    json={"observation": text}  # Changed from "text" to "observation"
                )
                response.raise_for_status()
                data = response.json()
                
                norme = data.get('norme', 'Aucune')
                self.print_success(f"Norme extraite: {norme}")
                print(f"Réponse complète: {json.dumps(data, indent=2, ensure_ascii=False)}")
                success_count += 1
            except requests.exceptions.HTTPError as e:
                error_detail = e.response.json() if e.response.text else str(e)
                self.print_error(f"Erreur HTTP {e.response.status_code}", json.dumps(error_detail, indent=2, ensure_ascii=False))
            except Exception as e:
                self.print_error(f"Erreur: {e}")
        
        result = f"✅ {success_count}/{len(test_cases)} réussis" if success_count == len(test_cases) else f"⚠️  {success_count}/{len(test_cases)} réussis"
        self.results.append(("Extraction Norme", result))
        return success_count == len(test_cases)
    
    def test_reformulate(self) -> bool:
        """Test du endpoint /api/v1/reformulate"""
        self.print_header("TEST 5: Reformulation d'Observation")
        
        test_cases = [
            {
                "text": "câble abîmé",
                "location": "Tableau principal"
            },
            {
                "text": "pas de protection différentielle"
            },
            {
                "text": "tableau pas aux normes",
                "location": "Garage"
            },
        ]
        
        success_count = 0
        for i, test_data in enumerate(test_cases, 1):
            self.print_info(f"Test 5.{i}: text='{test_data['text']}'")
            try:
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/reformulate",
                    json=test_data
                )
                response.raise_for_status()
                data = response.json()
                
                self.print_success(f"Reformulation reçue")
                print(f"  Original            : {test_data['text']}")
                print(f"  Reformulé           : {data.get('observation_corrigee', 'N/A')}")
                print(f"  Niveau gravité      : {data.get('niveau_gravite', 'N/A')}")
                print(f"  Délai recommandé    : {data.get('delai_recommande', 'N/A')}")
                print(f"  Norme applicable    : {data.get('norme_applicable', 'N/A')}")
                print(f"  Références normatives: {data.get('references_normatives', [])}")
                success_count += 1
            except requests.exceptions.HTTPError as e:
                error_detail = e.response.json() if e.response.text else str(e)
                self.print_error(f"Erreur HTTP {e.response.status_code}", json.dumps(error_detail, indent=2, ensure_ascii=False))
            except Exception as e:
                self.print_error(f"Erreur: {e}")
        
        result = f"✅ {success_count}/{len(test_cases)} réussis" if success_count == len(test_cases) else f"⚠️  {success_count}/{len(test_cases)} réussis"
        self.results.append(("Reformulation", result))
        return success_count == len(test_cases)
    
    def test_search(self) -> bool:
        """Test du endpoint /api/v1/search"""
        self.print_header("TEST 6: Recherche Sémantique")
        
        test_cases = [
            {
                "query": "protection différentielle 30mA",
                "max_results": 5
            },
            {
                "query": "mise à la terre",
                "max_results": 3
            },
            {
                "query": "tableau électrique",
                "max_results": 5
            },
        ]
        
        success_count = 0
        for i, test_data in enumerate(test_cases, 1):
            self.print_info(f"Test 6.{i}: query='{test_data['query']}', max_results={test_data['max_results']}")
            try:
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/search",
                    json=test_data
                )
                response.raise_for_status()
                data = response.json()
                
                results = data.get('results', [])
                self.print_success(f"Résultats trouvés: {len(results)}")
                
                if results:
                    print(f"  Top 3 résultats:")
                    for j, res in enumerate(results[:3], 1):
                        score = res.get('score', 0)
                        text = res.get('text', 'N/A')[:80]
                        print(f"    {j}. [Score: {score:.3f}] {text}...")
                
                success_count += 1
            except Exception as e:
                self.print_error(f"Erreur: {e}")
        
        result = f"✅ {success_count}/{len(test_cases)} réussis" if success_count == len(test_cases) else f"⚠️  {success_count}/{len(test_cases)} réussis"
        self.results.append(("Recherche Sémantique", result))
        return success_count == len(test_cases)
    
    def test_api_health_endpoint(self) -> bool:
        """Test du endpoint /api/v1/health"""
        self.print_header("TEST 7: API Health Check")
        try:
            response = self.session.get(f"{self.base_url}{API_PREFIX}/health")
            response.raise_for_status()
            data = response.json()
            
            self.print_success(f"Status: {response.status_code}")
            print(f"Réponse: {json.dumps(data, indent=2, ensure_ascii=False)}")
            
            self.results.append(("API Health", "✅ Réussi"))
            return True
        except Exception as e:
            self.print_error(f"Erreur: {e}")
            self.results.append(("API Health", f"❌ Échoué: {e}"))
            return False
    
    def print_summary(self):
        """Affiche un résumé des tests"""
        self.print_header("RÉSUMÉ DES TESTS")
        
        total = len(self.results)
        passed = sum(1 for _, result in self.results if "✅" in result)
        
        print(f"\nTotal des tests: {total}")
        print(f"Réussis: {passed}")
        print(f"Échoués: {total - passed}\n")
        
        for test_name, result in self.results:
            print(f"{test_name:.<50} {result}")
        
        if passed == total:
            print(f"\n🎉 TOUS LES TESTS SONT PASSÉS ! 🎉")
        else:
            print(f"\n⚠️  Certains tests ont échoué. Vérifiez les logs ci-dessus.")
    
    def run_all_tests(self):
        """Exécute tous les tests"""
        self.print_header("🚀 DÉMARRAGE DES TESTS API RAG")
        print(f"URL de base: {self.base_url}")
        
        start_time = time.time()
        
        # Exécution des tests
        self.test_health()
        self.test_status()
        self.test_autocomplete()
        self.test_extract_norme()
        self.test_reformulate()
        self.test_search()
        self.test_api_health_endpoint()
        
        end_time = time.time()
        duration = end_time - start_time
        
        # Afficher le résumé
        self.print_summary()
        print(f"\n⏱️  Durée totale: {duration:.2f} secondes")


def main():
    """Fonction principale"""
    print("\n╔═══════════════════════════════════════════════════════════════╗")
    print("║        TEST SUITE - API RAG NORMES ÉLECTRIQUES               ║")
    print("╚═══════════════════════════════════════════════════════════════╝\n")
    
    # Vérifier que l'API est accessible
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=5)
        response.raise_for_status()
        print(f"✅ API accessible à {BASE_URL}\n")
    except Exception as e:
        print(f"❌ Impossible de se connecter à l'API: {e}")
        print(f"Assurez-vous que l'API est démarrée avec:")
        print(f"uvicorn app:app --reload --host 0.0.0.0 --port 8000")
        return
    
    # Créer et exécuter le testeur
    tester = RAGAPITester(BASE_URL)
    tester.run_all_tests()


if __name__ == "__main__":
    main()

In [ ]:
"""
Tests avancés de l'API RAG avec des observations réelles - VERSION CORRIGÉE
Basé sur les exemples de votre base de données
FORMATS CONFIRMÉS : 
- Autocomplete: {"query": "..."}
- Extract Norme: {"observation": "..."}  
- Reformulate: {"text": "..."}
- Search: {"query": "...", "k": 5}
"""

import requests
import json
import time
from typing import List, Dict

BASE_URL = "http://localhost:8000"
API_PREFIX = "/api/v1"

# Observations réelles de votre base de données
REAL_OBSERVATIONS = [
    {
        "verbe": "Régler",
        "observation": "relais de protection Max de I de la canalisation contre les court-circuits",
        "norme_attendue": "R.4215-6/ Norme NF C 13-200 art 431-432"
    },
    {
        "verbe": "Remplacer",
        "observation": "fusible par un modèle calibré à",
        "norme_attendue": "R.4215-6/ Norme NF C 13-200 art 431-432"
    },
    {
        "verbe": "Régler",
        "observation": "relais de protection Max de I contre les surcharges à",
        "norme_attendue": "R.4215-6/ Norme NF C 13-200 art 431-432"
    },
    {
        "verbe": "Remplacer",
        "observation": "appareil de commande par un modèle adapté au courant nominal ou calibrer le dispositif de protection à",
        "norme_attendue": "R.4215-6/ Norme NF C 15-100 art 430-533"
    },
    {
        "verbe": "Réaliser",
        "observation": "coordination entre les dispositifs de protection de surcharge et de court-cutcut",
        "norme_attendue": "R.4215-6/ Norme NF C 15-100 art 435"
    },
    {
        "verbe": "Remplacer",
        "observation": "câble par un modèle dont la section de l'écran est dimensionné pour supporter l'intensité de court-circuit provoquée par le claquage de l'isolant entre la phase et l'armature",
        "norme_attendue": "R.4215-6/ Norme NF C 13-200 art 527-528"
    },
    {
        "verbe": "Régler",
        "observation": "relais de protection Max de I du transformateur contre les court-circuits",
        "norme_attendue": "R.4215-6/ Norme NF C 13-200 art 431-432"
    },
    {
        "verbe": "Protéger",
        "observation": "secondaire du transformateur contre les surcharges",
        "norme_attendue": "R.4215-6/ Norme NF C 13-100 art 432"
    },
    {
        "verbe": "Protéger",
        "observation": "sélectivement les circuits d'éclairage de chaque bloc opératoire",
        "norme_attendue": "R.4215-6/ Norme NF C 15-211 art 9"
    },
]


class AdvancedRAGTester:
    """Testeur avancé pour l'API RAG - VERSION CORRIGÉE"""
    
    def __init__(self, base_url: str = BASE_URL):
        self.base_url = base_url
        self.session = requests.Session()
        self.results = []
    
    def print_header(self, text: str, char: str = "="):
        """Affiche un header"""
        print(f"\n{char*80}")
        print(f"{text.center(80)}")
        print(f"{char*80}")
    
    def print_test_result(self, test_num: int, success: bool, details: str = ""):
        """Affiche le résultat d'un test"""
        icon = "✅" if success else "❌"
        print(f"{icon} Test {test_num}: {details}")
    
    def test_autocomplete_real_cases(self) -> Dict:
        """Test l'autocomplétion avec des verbes et débuts réels - FORMAT CORRIGÉ"""
        self.print_header("TEST AUTOCOMPLÉTION - CAS RÉELS", "=")
        
        test_cases = [
            {"query": "Régler relais de protection"},
            {"query": "Remplacer fusible par"},
            {"query": "Protéger secondaire du"},
            {"query": "Réaliser coordination entre"},
            {"query": "Protéger sélectivement les circuits"},
        ]
        
        results = {"total": len(test_cases), "passed": 0, "failed": 0, "details": []}
        
        for i, case in enumerate(test_cases, 1):
            query = case['query']
            
            try:
                # FORMAT CORRIGÉ : {"query": "..."} sans "max_results"
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/autocomplete",
                    json={"query": query},
                    timeout=10
                )
                
                if response.status_code == 200:
                    data = response.json()
                    suggestions = data.get('suggestions', [])
                    count = data.get('count', 0)
                    
                    results['passed'] += 1
                    results['details'].append({
                        "test": i,
                        "query": query,
                        "success": True,
                        "count": count,
                        "top_suggestions": suggestions[:3]
                    })
                    
                    print(f"\n✅ Test {i}: '{query}'")
                    print(f"   📊 {count} suggestions trouvées")
                    if suggestions:
                        print(f"   🔝 Top 3:")
                        for j, sugg in enumerate(suggestions[:3], 1):
                            print(f"      {j}. {sugg[:80]}...")
                else:
                    results['failed'] += 1
                    error_detail = response.json() if response.text else str(response.status_code)
                    print(f"\n❌ Test {i}: Erreur {response.status_code}")
                    print(f"   Détail: {json.dumps(error_detail, indent=2)[:200]}...")
                    
            except Exception as e:
                results['failed'] += 1
                print(f"\n❌ Test {i}: Exception - {e}")
        
        return results
    
    def test_extract_norme_real_cases(self) -> Dict:
        """Test l'extraction de normes avec des observations réelles - FORMAT CORRIGÉ"""
        self.print_header("TEST EXTRACTION NORME - CAS RÉELS", "=")
        
        results = {"total": len(REAL_OBSERVATIONS), "passed": 0, "failed": 0, "correct_norms": 0, "details": []}
        
        for i, obs_data in enumerate(REAL_OBSERVATIONS, 1):
            observation = f"{obs_data['verbe']} {obs_data['observation']}"
            norme_attendue = obs_data['norme_attendue']
            
            try:
                # FORMAT CORRIGÉ : {"observation": "..."}
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/extract_norme",
                    json={"observation": observation},
                    timeout=10
                )
                
                if response.status_code == 200:
                    data = response.json()
                    norme_extraite = data.get('norme', '')
                    confidence = data.get('confidence', 0)
                    
                    # Vérifier si la norme extraite correspond
                    norme_match = norme_attendue.lower() in norme_extraite.lower() or \
                                 any(part in norme_extraite.lower() for part in norme_attendue.lower().split('/'))
                    
                    results['passed'] += 1
                    if norme_match:
                        results['correct_norms'] += 1
                    
                    results['details'].append({
                        "test": i,
                        "observation": observation[:60],
                        "norme_attendue": norme_attendue,
                        "norme_extraite": norme_extraite,
                        "confidence": confidence,
                        "match": norme_match
                    })
                    
                    match_icon = "✅" if norme_match else "⚠️"
                    print(f"\n{match_icon} Test {i}:")
                    print(f"   📝 Observation: {observation[:60]}...")
                    print(f"   🎯 Attendu    : {norme_attendue}")
                    print(f"   🤖 Extrait    : {norme_extraite[:80]}...")
                    print(f"   📊 Confiance  : {confidence:.2f}")
                else:
                    results['failed'] += 1
                    error_detail = response.json() if response.text else str(response.status_code)
                    print(f"\n❌ Test {i}: Erreur {response.status_code}")
                    print(f"   Détail: {json.dumps(error_detail, indent=2)[:200]}...")
                    
            except Exception as e:
                results['failed'] += 1
                print(f"\n❌ Test {i}: Exception - {e}")
        
        return results
    
    def test_reformulate_real_cases(self) -> Dict:
        """Test la reformulation avec des observations réelles - FORMAT CORRIGÉ"""
        self.print_header("TEST REFORMULATION - CAS RÉELS", "=")
        
        observations_to_reformulate = [
            "relais de protection Max de I contre les surcharges mal réglé",
            "fusible non calibré correctement",
            "câble dont section écran insuffisante pour court-circuit",
            "protection secondaire transformateur absente",
            "pas de protection sélective éclairage bloc opératoire",
        ]
        
        results = {"total": len(observations_to_reformulate), "passed": 0, "failed": 0, "details": []}
        
        for i, text in enumerate(observations_to_reformulate, 1):
            try:
                # FORMAT CORRIGÉ : {"text": "..."} seulement
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/reformulate",
                    json={"text": text},
                    timeout=15
                )
                
                if response.status_code == 200:
                    data = response.json()
                    
                    results['passed'] += 1
                    results['details'].append({
                        "test": i,
                        "original": text,
                        "reformule": data.get('observation_corrigee', ''),
                        "gravite": data.get('niveau_gravite', ''),
                        "delai": data.get('delai_recommande', ''),
                        "norme": data.get('norme_applicable', '')
                    })
                    
                    print(f"\n✅ Test {i}:")
                    print(f"   📝 Original       : {text}")
                    print(f"   ✏️  Reformulé      : {data.get('observation_corrigee', 'N/A')[:80]}...")
                    print(f"   ⚠️  Gravité        : {data.get('niveau_gravite', 'N/A')}")
                    print(f"   ⏱️  Délai          : {data.get('delai_recommande', 'N/A')}")
                    print(f"   📋 Norme          : {data.get('norme_applicable', 'N/A')[:60]}...")
                    
                    refs = data.get('references_normatives', [])
                    if refs:
                        print(f"   📚 Références     : {', '.join(refs[:2])}")
                else:
                    results['failed'] += 1
                    error_detail = response.json() if response.text else str(response.status_code)
                    print(f"\n❌ Test {i}: Erreur {response.status_code}")
                    print(f"   Détail: {json.dumps(error_detail, indent=2)[:200]}...")
                    
            except Exception as e:
                results['failed'] += 1
                print(f"\n❌ Test {i}: Exception - {e}")
        
        return results
    
    def test_search_technical_terms(self) -> Dict:
        """Test la recherche avec des termes techniques précis - FORMAT CORRIGÉ"""
        self.print_header("TEST RECHERCHE - TERMES TECHNIQUES", "=")
        
        technical_queries = [
            "relais de protection Max de I contre court-circuits",
            "fusible calibré dispositif protection",
            "coordination dispositifs protection surcharge",
            "section écran câble court-circuit",
            "protection sélective circuits éclairage",
            "transformateur protection surcharges",
        ]
        
        results = {"total": len(technical_queries), "passed": 0, "failed": 0, "details": []}
        
        for i, query in enumerate(technical_queries, 1):
            try:
                # FORMAT CORRIGÉ : {"query": "...", "k": 5}
                response = self.session.post(
                    f"{self.base_url}{API_PREFIX}/search",
                    json={"query": query, "k": 5},
                    timeout=10
                )
                
                if response.status_code == 200:
                    data = response.json()
                    search_results = data.get('results', [])
                    count = len(search_results)
                    
                    results['passed'] += 1
                    results['details'].append({
                        "test": i,
                        "query": query,
                        "count": count,
                        "top_score": search_results[0].get('score', 0) if search_results else 0
                    })
                    
                    print(f"\n✅ Test {i}: '{query}'")
                    print(f"   📊 {count} résultats trouvés")
                    
                    if search_results:
                        print(f"   🔝 Top résultat:")
                        top = search_results[0]
                        print(f"      Score: {top.get('score', 0):.3f}")
                        print(f"      Texte: {top.get('content', top.get('text', 'N/A'))[:100]}...")
                        if top.get('norme'):
                            print(f"      Norme: {top.get('norme')}")
                else:
                    results['failed'] += 1
                    error_detail = response.json() if response.text else str(response.status_code)
                    print(f"\n❌ Test {i}: Erreur {response.status_code}")
                    print(f"   Détail: {json.dumps(error_detail, indent=2)[:200]}...")
                    
            except Exception as e:
                results['failed'] += 1
                print(f"\n❌ Test {i}: Exception - {e}")
        
        return results
    
    def print_final_summary(self, all_results: Dict):
        """Affiche un résumé final complet"""
        self.print_header("📊 RÉSUMÉ GÉNÉRAL DES TESTS AVANCÉS", "=")
        
        total_tests = sum(r['total'] for r in all_results.values())
        total_passed = sum(r['passed'] for r in all_results.values())
        total_failed = sum(r['failed'] for r in all_results.values())
        
        print(f"\n🎯 Statistiques globales:")
        print(f"   Total tests       : {total_tests}")
        print(f"   ✅ Réussis        : {total_passed}")
        print(f"   ❌ Échoués        : {total_failed}")
        print(f"   📈 Taux réussite  : {(total_passed/total_tests*100):.1f}%" if total_tests > 0 else "0%")
        
        print(f"\n📋 Détail par catégorie:")
        
        # Autocomplétion
        ac = all_results.get('autocomplete', {})
        print(f"\n   🔤 AUTOCOMPLÉTION:")
        print(f"      Tests: {ac.get('passed', 0)}/{ac.get('total', 0)} réussis")
        
        # Extraction de normes
        en = all_results.get('extract_norme', {})
        print(f"\n   📜 EXTRACTION DE NORME:")
        print(f"      Tests: {en.get('passed', 0)}/{en.get('total', 0)} réussis")
        if 'correct_norms' in en and en['total'] > 0:
            print(f"      Normes correctes: {en['correct_norms']}/{en['total']} ({(en['correct_norms']/en['total']*100):.1f}%)")
        
        # Reformulation
        rf = all_results.get('reformulate', {})
        print(f"\n   ✏️  REFORMULATION:")
        print(f"      Tests: {rf.get('passed', 0)}/{rf.get('total', 0)} réussis")
        
        # Recherche
        sr = all_results.get('search', {})
        print(f"\n   🔍 RECHERCHE SÉMANTIQUE:")
        print(f"      Tests: {sr.get('passed', 0)}/{sr.get('total', 0)} réussis")
        
        if total_passed == total_tests:
            print(f"\n{'🎉'*40}")
            print(f"{'TOUS LES TESTS AVANCÉS SONT PASSÉS !'.center(80)}")
            print(f"{'🎉'*40}")
        else:
            print(f"\n⚠️  {total_failed} test(s) ont échoué. Consultez les détails ci-dessus.")
    
    def run_all_advanced_tests(self):
        """Exécute tous les tests avancés"""
        print("\n╔═══════════════════════════════════════════════════════════════╗")
        print("║         TESTS AVANCÉS - API RAG NORMES ÉLECTRIQUES           ║")
        print("║              Basés sur des Données Réelles                   ║")
        print("╚═══════════════════════════════════════════════════════════════╝")
        
        start_time = time.time()
        
        all_results = {}
        
        # Test 1: Autocomplétion (format corrigé)
        all_results['autocomplete'] = self.test_autocomplete_real_cases()
        
        # Test 2: Extraction de normes (format corrigé)
        all_results['extract_norme'] = self.test_extract_norme_real_cases()
        
        # Test 3: Reformulation (format corrigé)
        all_results['reformulate'] = self.test_reformulate_real_cases()
        
        # Test 4: Recherche (format corrigé)
        all_results['search'] = self.test_search_technical_terms()
        
        end_time = time.time()
        duration = end_time - start_time
        
        # Résumé final
        self.print_final_summary(all_results)
        
        print(f"\n⏱️  Durée totale: {duration:.2f} secondes")
        print(f"\n{'='*80}\n")


def main():
    """Fonction principale"""
    
    # Vérifier que l'API est accessible
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=5)
        response.raise_for_status()
        print(f"✅ API accessible à {BASE_URL}\n")
    except Exception as e:
        print(f"❌ Impossible de se connecter à l'API: {e}")
        print(f"\n💡 Assurez-vous que l'API est démarrée avec:")
        print(f"uvicorn app:app --reload --host 0.0.0.0 --port 8000")
        return
    
    # Créer et exécuter le testeur
    tester = AdvancedRAGTester(BASE_URL)
    tester.run_all_advanced_tests()


if __name__ == "__main__":
    main()

In [ ]:
# =============================
# 🚀 RAG System pour Google Colab
# =============================

# 1️⃣ Vérifier le GPU
import torch
print("🔍 Vérification GPU...")
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 2️⃣ Installation (décommenter si nécessaire)
# !pip install -q transformers accelerate sentence-transformers faiss-cpu langchain-community langchain-core

# 3️⃣ Imports
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
import torch
import gc

# =============================
# 🎯 Configuration - Modèles pour Colab
# =============================
models_colab = {
    "tiny": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",      # ~1GB - Ultra rapide
    "phi": "microsoft/phi-2",                           # ~3GB - Bon équilibre
    "qwen": "Qwen/Qwen2.5-3B-Instruct",                # ~4GB - Excellent
    "gemma": "google/gemma-2b-it",                     # ~2GB - Rapide
}

# 👇 Change ici (tiny/phi/qwen/gemma)
MODEL_CHOICE = "phi"  # Recommandé pour Colab gratuit

model_name = models_colab[MODEL_CHOICE]
print(f"\n🤖 Modèle sélectionné : {model_name}\n")

# =============================
# 📥 Chargement du modèle
# =============================
print("⏳ Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ Chargement du modèle...")

# Configuration optimale pour Colab
if torch.cuda.is_available():
    # Sur GPU : utiliser float16 pour économiser la mémoire
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    print("✅ Modèle chargé sur GPU (float16)")
else:
    # Sur CPU (moins recommandé mais fonctionne)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True
    )
    print("✅ Modèle chargé sur CPU")

# Nettoyer la mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =============================
# 🔍 Setup embeddings + FAISS
# =============================
print("\n⏳ Chargement des embeddings...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# Documents de démonstration (à remplacer par tes données)
documents = [
    Document(page_content="Python est un langage de programmation interprété, orienté objet et de haut niveau."),
    Document(page_content="Les Large Language Models (LLMs) comme GPT, Claude et Mistral utilisent l'architecture Transformer."),
    Document(page_content="RAG (Retrieval Augmented Generation) améliore les LLMs en ajoutant une base de connaissances."),
    Document(page_content="FAISS est une bibliothèque développée par Meta pour la recherche de similarité vectorielle."),
    Document(page_content="Google Colab offre des GPUs gratuits pour l'entraînement et l'inférence de modèles ML."),
    Document(page_content="Transformers est la bibliothèque principale de HuggingFace pour utiliser des modèles pré-entraînés."),
]

print("⏳ Création de l'index FAISS...")
vector_store = FAISS.from_documents(documents, embedding_model)
print("✅ Index FAISS créé avec succès!")

# =============================
# 🧠 Fonction RAG
# =============================
def rag_query(query, k=3, max_tokens=200):
    """
    Effectue une requête RAG
    
    Args:
        query: Question à poser
        k: Nombre de documents à récupérer
        max_tokens: Longueur max de la réponse
    """
    # Recherche de documents pertinents
    relevant_docs = vector_store.similarity_search(query, k=k)
    
    # Construction du contexte
    context = "\n\n".join([f"Document {i+1}: {doc.page_content}" 
                           for i, doc in enumerate(relevant_docs)])
    
    # Construction du prompt
    prompt = f"""Tu es un assistant qui répond aux questions en te basant sur les documents fournis.

Documents de référence :
{context}

Question : {query}

Réponds de manière concise et précise en te basant uniquement sur les documents ci-dessus."""

    # Tokenisation
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=1024
    )
    
    # Déplacer sur le bon device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Génération
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Décodage
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire uniquement la réponse (après le prompt)
    if "Réponds de manière concise" in full_response:
        response = full_response.split("Réponds de manière concise")[1].strip()
    else:
        response = full_response
    
    return {
        "query": query,
        "response": response,
        "sources": relevant_docs
    }

# =============================
# 🧪 Tests automatiques
# =============================
print("\n" + "="*60)
print("🚀 SYSTÈME RAG PRÊT - TESTS AUTOMATIQUES")
print("="*60 + "\n")

test_questions = [
    "Qu'est-ce que Python ?",
    "Explique-moi ce qu'est RAG",
    "Qu'est-ce que Google Colab offre ?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}/{len(test_questions)}")
    print(f"{'='*60}")
    print(f"❓ Question : {question}")
    
    result = rag_query(question)
    
    print(f"\n💡 Réponse :")
    print(result['response'][:400])  # Limiter l'affichage
    
    print(f"\n📚 Sources utilisées :")
    for j, doc in enumerate(result['sources'], 1):
        print(f"  [{j}] {doc.page_content[:80]}...")
    
    print()

# =============================
# 💬 Mode interactif
# =============================
print("\n" + "="*60)
print("💬 MODE INTERACTIF")
print("="*60)
print("Tapez votre question ou 'quit' pour quitter\n")

while True:
    try:
        user_input = input("🔍 Votre question : ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'q', '']:
            print("👋 Au revoir !")
            break
        
        result = rag_query(user_input)
        
        print(f"\n✨ Réponse :")
        print(result['response'])
        
        print(f"\n📚 Sources :")
        for i, doc in enumerate(result['sources'], 1):
            print(f"  [{i}] {doc.page_content[:100]}...")
        print()
        
    except KeyboardInterrupt:
        print("\n👋 Au revoir !")
        break
    except Exception as e:
        print(f"❌ Erreur : {e}")

# =============================
# 📝 Fonction pour ajouter des documents
# =============================
def add_documents(texts):
    """
    Ajoute de nouveaux documents à la base FAISS
    
    Usage:
        add_documents([
            "Nouveau texte 1",
            "Nouveau texte 2"
        ])
    """
    new_docs = [Document(page_content=text) for text in texts]
    vector_store.add_documents(new_docs)
    print(f"✅ {len(texts)} document(s) ajouté(s) à la base !")
    return True

# =============================
# 💾 Sauvegarder/Charger l'index
# =============================
def save_index(path="faiss_index"):
    """Sauvegarde l'index FAISS"""
    vector_store.save_local(path)
    print(f"✅ Index sauvegardé dans '{path}'")

def load_index(path="faiss_index"):
    """Charge un index FAISS existant"""
    global vector_store
    vector_store = FAISS.load_local(path, embedding_model, allow_dangerous_deserialization=True)
    print(f"✅ Index chargé depuis '{path}'")

print("\n" + "="*60)
print("📖 FONCTIONS DISPONIBLES :")
print("="*60)
print("• rag_query(question)           - Poser une question")
print("• add_documents([textes])       - Ajouter des documents")
print("• save_index('mon_index')       - Sauvegarder l'index")
print("• load_index('mon_index')       - Charger un index")
print("="*60)

In [ ]:
"""
Configuration LLM Local pour remplacer Groq
Compatible avec ton système RAG existant
Version: 1.0 - Décembre 2024
"""
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.language_models import BaseLLM
from typing import Optional, List
import gc

# =============================================================================
# CONFIGURATION CENTRALISÉE
# =============================================================================

LOCAL_LLM_CONFIG = {
    "model": "Qwen/Qwen2.5-3B-Instruct",  # Modèle par défaut
    "temperature": 0,
    "max_tokens": 2048,
    "device": "auto",  # "cuda", "cpu", ou "auto"
    "load_in_8bit": False,  # Mettre True si mémoire limitée
}

# =============================================================================
# MODÈLES LOCAUX DISPONIBLES
# =============================================================================

AVAILABLE_LOCAL_MODELS = {
    "recommandes": {
        "qwen-3b": "Qwen/Qwen2.5-3B-Instruct",           # 3B - Excellent équilibre
        "phi-2": "microsoft/phi-2",                       # 2.7B - Rapide
        "gemma-2b": "google/gemma-2b-it",                # 2B - Compact
    },
    "puissants": {
        "qwen-7b": "Qwen/Qwen2.5-7B-Instruct",           # 7B - Très bon
        "mistral-7b": "mistralai/Mistral-7B-Instruct-v0.2",  # 7B - Référence
    },
    "ultra_legers": {
        "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",  # 1.1B - Ultra rapide
    }
}

# =============================================================================
# CLIENT LLM LOCAL - COMPATIBLE AVEC TON CODE GROQ
# =============================================================================

class LocalLLMClient:
    """
    Client LLM Local qui remplace GroqLLMClient
    Interface compatible avec ton code existant
    """
    
    def __init__(
        self, 
        model_name: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        device: Optional[str] = None,
        load_in_8bit: bool = False
    ):
        """
        Initialise le client LLM local
        
        Args:
            model_name: Nom du modèle HuggingFace
            temperature: Température pour la génération (0-1)
            max_tokens: Nombre maximum de tokens
            device: "cuda", "cpu", ou "auto"
            load_in_8bit: Activer quantification 8-bit (économise mémoire)
        """
        # Configuration
        self.model_name = model_name or LOCAL_LLM_CONFIG["model"]
        self.temperature = temperature if temperature is not None else LOCAL_LLM_CONFIG["temperature"]
        self.max_tokens = max_tokens or LOCAL_LLM_CONFIG["max_tokens"]
        self.device = device or LOCAL_LLM_CONFIG["device"]
        self.load_in_8bit = load_in_8bit or LOCAL_LLM_CONFIG["load_in_8bit"]
        
        print(f"🤖 Chargement du modèle local: {self.model_name}")
        
        # Détecter le device optimal
        if self.device == "auto":
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        print(f"📍 Device: {self.device}")
        
        # Charger le tokenizer
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )
            
            # Ajouter padding token si nécessaire
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            
            print("✅ Tokenizer chargé")
            
        except Exception as e:
            raise ValueError(f"❌ Erreur tokenizer: {e}")
        
        # Charger le modèle
        try:
            load_kwargs = {
                "trust_remote_code": True,
                "low_cpu_mem_usage": True,
            }
            
            # Configuration selon device
            if self.device == "cuda":
                if self.load_in_8bit:
                    load_kwargs["load_in_8bit"] = True
                    load_kwargs["device_map"] = "auto"
                    print("⚡ Chargement en 8-bit (GPU)")
                else:
                    load_kwargs["torch_dtype"] = torch.float16
                    load_kwargs["device_map"] = "auto"
                    print("⚡ Chargement en float16 (GPU)")
            else:
                load_kwargs["torch_dtype"] = torch.float32
                print("⚡ Chargement sur CPU")
            
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                **load_kwargs
            )
            
            # Déplacer sur device si nécessaire
            if self.device == "cpu":
                self.model = self.model.to(self.device)
            
            print("✅ Modèle chargé avec succès!")
            
            # Nettoyer mémoire
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
        except Exception as e:
            raise ValueError(f"❌ Erreur chargement modèle: {e}")
        
        # Créer le pipeline HuggingFace
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=self.max_tokens,
            temperature=self.temperature,
            top_p=0.95,
            do_sample=True if self.temperature > 0 else False,
            repetition_penalty=1.15,
        )
        
        # Wrapper LangChain (compatible avec ton code)
        self.llm = HuggingFacePipeline(pipeline=self.pipeline)
        
        print("✅ LocalLLMClient prêt!\n")
    
    def invoke(self, prompt: str) -> str:
        """
        Génère une réponse (compatible avec Groq)
        
        Args:
            prompt: Le prompt d'entrée
            
        Returns:
            str: La réponse générée
        """
        try:
            # Utiliser le pipeline directement
            result = self.pipeline(
                prompt,
                max_new_tokens=self.max_tokens,
                temperature=self.temperature,
                return_full_text=False,  # Retourner seulement la génération
            )
            
            response = result[0]["generated_text"]
            return response.strip()
            
        except Exception as e:
            print(f"❌ Erreur invoke: {e}")
            raise
    
    def generate(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """
        Alias pour invoke() (compatibilité)
        """
        return self.invoke(prompt)
    
    def batch_invoke(self, prompts: List[str]) -> List[str]:
        """
        Traite plusieurs prompts
        """
        return [self.invoke(p) for p in prompts]
    
    def stream(self, prompt: str):
        """
        Génération en streaming (non implémenté pour l'instant)
        """
        # Pour streaming, utiliser TextIteratorStreamer de transformers
        raise NotImplementedError("Streaming non disponible pour modèles locaux")
    
    def __del__(self):
        """Nettoyage mémoire"""
        if hasattr(self, 'model'):
            del self.model
        if hasattr(self, 'tokenizer'):
            del self.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# =============================================================================
# ALIAS POUR COMPATIBILITÉ AVEC TON CODE EXISTANT
# =============================================================================

# Remplace GroqLLMClient par LocalLLMClient dans ton code
GroqLLMClient = LocalLLMClient

# =============================================================================
# FONCTIONS UTILITAIRES
# =============================================================================

def list_available_models():
    """Affiche tous les modèles locaux disponibles"""
    print("\n🤖 MODÈLES LOCAUX DISPONIBLES")
    print("=" * 60)
    
    for category, models in AVAILABLE_LOCAL_MODELS.items():
        print(f"\n📂 {category.upper()}")
        print("-" * 40)
        for name, model_path in models.items():
            print(f"  ✓ {name:15} → {model_path}")
    
    print(f"\n💡 Modèle par défaut: {LOCAL_LLM_CONFIG['model']}")
    print(f"🔧 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

def test_connection():
    """
    Test le modèle local
    """
    try:
        print("🔄 Test du modèle local...")
        client = LocalLLMClient()
        response = client.invoke("Réponds uniquement par 'OK'")
        print(f"✅ Test réussi! Réponse: {response[:50]}")
        return True
    except Exception as e:
        print(f"❌ Échec du test: {e}")
        return False

def get_model_info(model_key: str) -> dict:
    """
    Retourne les informations sur un modèle
    """
    model_info = {
        "qwen-3b": {
            "description": "Qwen 2.5 3B - Excellent équilibre performance/taille",
            "use_case": "Production, analyses complexes",
            "ram_required": "~4GB",
            "speed": "⚡⚡⚡⚡"
        },
        "phi-2": {
            "description": "Microsoft Phi-2 - 2.7B paramètres",
            "use_case": "Tâches variées, rapide",
            "ram_required": "~3GB",
            "speed": "⚡⚡⚡⚡⚡"
        },
        "mistral-7b": {
            "description": "Mistral 7B - Référence qualité",
            "use_case": "Meilleure qualité possible",
            "ram_required": "~8GB",
            "speed": "⚡⚡⚡"
        },
        "tinyllama": {
            "description": "TinyLlama 1.1B - Ultra compact",
            "use_case": "Tests rapides, ressources limitées",
            "ram_required": "~1.5GB",
            "speed": "⚡⚡⚡⚡⚡"
        }
    }
    
    return model_info.get(model_key, {"description": "Modèle non documenté"})

def benchmark_model(model_name: str = None, num_tests: int = 3):
    """
    Benchmark simple d'un modèle
    """
    import time
    
    print(f"\n⏱️ BENCHMARK DU MODÈLE")
    print("=" * 60)
    
    client = LocalLLMClient(model_name=model_name)
    
    test_prompts = [
        "Qu'est-ce que la norme NF C 15-100 ?",
        "Explique brièvement la section des conducteurs.",
        "Définis un tableau électrique."
    ]
    
    times = []
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\nTest {i}/{num_tests}: {prompt[:40]}...")
        start = time.time()
        response = client.invoke(prompt)
        elapsed = time.time() - start
        times.append(elapsed)
        print(f"⏱️ Temps: {elapsed:.2f}s")
        print(f"📝 Réponse: {response[:100]}...")
    
    avg_time = sum(times) / len(times)
    print(f"\n📊 Temps moyen: {avg_time:.2f}s")
    
    del client
    gc.collect()

# =============================================================================
# GUIDE DE MIGRATION
# =============================================================================

def print_migration_guide():
    """
    Affiche le guide pour migrer de Groq vers Local
    """
    print("""
╔══════════════════════════════════════════════════════════════╗
║          GUIDE DE MIGRATION : GROQ → LOCAL                   ║
╚══════════════════════════════════════════════════════════════╝

📋 ÉTAPES DE MIGRATION :

1️⃣ REMPLACER L'IMPORT
   ❌ Ancien: from groq_config import GroqLLMClient
   ✅ Nouveau: from local_llm_config import LocalLLMClient

2️⃣ REMPLACER L'INITIALISATION
   ❌ Ancien: client = GroqLLMClient(api_key="gsk_...")
   ✅ Nouveau: client = LocalLLMClient(model_name="Qwen/Qwen2.5-3B-Instruct")

3️⃣ UTILISATION IDENTIQUE
   client.invoke("Ton prompt")        # Fonctionne pareil !
   client.generate("Ton prompt")      # Fonctionne pareil !
   client.batch_invoke([...])         # Fonctionne pareil !

4️⃣ FICHIERS À MODIFIER
   - main_pipeline.py
   - rag_pipeline.py (si existe)
   - app1.py, app2.py
   - Tout fichier utilisant GroqLLMClient

5️⃣ EXEMPLE DE MODIFICATION
   # Dans main_pipeline.py ou autres fichiers
   
   # Ligne à changer :
   from groq_config import GroqLLMClient
   
   # Devient :
   from local_llm_config import LocalLLMClient as GroqLLMClient
   
   # Le reste du code reste IDENTIQUE !

╔══════════════════════════════════════════════════════════════╗
║                    AVANTAGES                                 ║
╚══════════════════════════════════════════════════════════════╝
✅ Plus besoin de clé API
✅ Fonctionne hors ligne
✅ Pas de limite de requêtes
✅ Gratuit après téléchargement
✅ Données 100% privées

╔══════════════════════════════════════════════════════════════╗
║                    INCONVÉNIENTS                             ║
╚══════════════════════════════════════════════════════════════╝
⚠️ Plus lent que Groq (surtout sur CPU)
⚠️ Nécessite RAM/GPU local
⚠️ Téléchargement initial des modèles (~3-7GB)
    """)

# =============================================================================
# TEST RAPIDE
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 LOCAL LLM CONFIG - REMPLACEMENT DE GROQ")
    print("="*60)
    
    # Afficher les modèles
    list_available_models()
    
    # Afficher le guide de migration
    print_migration_guide()
    
    # Test de connexion
    print("\n" + "="*60)
    print("🔧 TEST DU MODÈLE LOCAL")
    print("="*60)
    
    choice = input("\n🤔 Lancer le test ? (o/n): ").lower()
    if choice == 'o':
        test_connection()
        
        bench = input("\n🤔 Lancer le benchmark ? (o/n): ").lower()
        if bench == 'o':
            benchmark_model()

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# Définir le token dans l'environnement
os.environ["HF_TOKEN"] = "hf_..."  # Ton token

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
local_dir = "llm_model/mistral-7b"

tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    cache_dir=local_dir,
    use_auth_token=True,  # Utilise automatiquement HF_TOKEN
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=local_dir,
    use_auth_token=True,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype="auto"
)

print("✅ Mistral 7B téléchargé et prêt dans:", local_dir)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py:1041: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# Définir le token dans l'environnement (ou le passer directement)
os.environ["HF_TOKEN"] = "hf_..."  # Remplace par ton token

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
local_dir = "/home/user/models/mistral-7b"

# Récupérer le token
hf_token = os.environ.get("HF_TOKEN")

# Télécharger le tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    cache_dir=local_dir,
    token=hf_token,  # ✅ Utilise 'token' au lieu de 'use_auth_token'
    trust_remote_code=True
)

# Télécharger le modèle
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=local_dir,
    token=hf_token,  # ✅ Utilise 'token' au lieu de 'use_auth_token'
    trust_remote_code=True,
    device_map="auto",
    torch_dtype="auto"
)

print("✅ Mistral 7B téléchargé et prêt dans:", local_dir)

In [ ]:
print("Hello World")

In [ ]:
def chat_mistral(prompt, max_tokens=200, temperature=0.7):
    """
    Fonction d'inférence pour Mistral-7B-Instruct
    """
    # Format Mistral-Instruct
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    
    # Tokenizer
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Génération
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Décoder seulement la réponse (sans le prompt)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Exemples d'utilisation
print("🤖 Réponse 1:")
print(chat_mistral("Quelle est la capitale de la France ?"))

print("\n🤖 Réponse 2:")
print(chat_mistral("Explique-moi la physique quantique en 3 phrases."))

print("\n🤖 Réponse 3:")
print(chat_mistral("Écris un haiku sur le printemps."))

In [ ]:
"""
Version simplifiée et corrigée pour charger Mistral-7B
"""
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

# Configuration
LOCAL_DIR = "/home/user/models/mistral-7b"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

def find_model_path():
    """
    Trouve le bon chemin vers le modèle dans le cache HuggingFace
    """
    print("🔍 Recherche du modèle dans le cache...")
    
    cache_path = Path(LOCAL_DIR)
    
    # Méthode 1: Chercher dans models--mistralai--Mistral-7B-Instruct-v0.2
    model_cache = cache_path / "models--mistralai--Mistral-7B-Instruct-v0.2"
    if model_cache.exists():
        print(f"✓ Trouvé: {model_cache}")
        
        # Chercher le snapshot le plus récent
        snapshots_dir = model_cache / "snapshots"
        if snapshots_dir.exists():
            versions = [d for d in snapshots_dir.iterdir() if d.is_dir()]
            if versions:
                latest = max(versions, key=lambda p: p.stat().st_mtime)
                print(f"✓ Snapshot: {latest.name}")
                return str(latest)
    
    # Méthode 2: Chercher n'importe quel snapshot
    snapshots = list(cache_path.rglob("snapshots/*/config.json"))
    if snapshots:
        snapshot_dir = snapshots[0].parent
        print(f"✓ Snapshot trouvé: {snapshot_dir}")
        return str(snapshot_dir)
    
    # Méthode 3: Si rien trouvé, retourner le nom du modèle (téléchargement)
    print(f"⚠️  Aucun cache trouvé, utilisera: {MODEL_NAME}")
    return MODEL_NAME

def load_mistral():
    """
    Charge Mistral-7B de manière robuste
    """
    print("\n" + "="*60)
    print("🚀 CHARGEMENT DE MISTRAL-7B")
    print("="*60)
    
    # Trouver le bon chemin
    model_path = find_model_path()
    print(f"\n📂 Chemin utilisé: {model_path}")
    
    try:
        # Charger le tokenizer
        print("\n1️⃣ Chargement du tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            legacy=False,
            trust_remote_code=True
        )
        print("   ✅ Tokenizer OK")
        
        # Charger le modèle
        print("\n2️⃣ Chargement du modèle (peut prendre 1-2 min)...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        print("   ✅ Modèle OK")
        
        # Device
        device = next(model.parameters()).device
        print(f"\n3️⃣ Device: {device}")
        
        return tokenizer, model, device
        
    except Exception as e:
        print(f"\n❌ Erreur: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

def test_inference(tokenizer, model, device):
    """
    Test d'inférence simple
    """
    print("\n" + "="*60)
    print("🧪 TEST D'INFÉRENCE")
    print("="*60)
    
    prompt = "<s>[INST] Quelle est la capitale de la France ? Réponds en un mot. [/INST]"
    
    print(f"\n📝 Prompt: {prompt}")
    print("\n🔄 Génération en cours...")
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    
    print(f"\n💬 Réponse: {response.strip()}")
    print("\n✅ TEST RÉUSSI!")

def create_client_class(tokenizer, model, device):
    """
    Crée une classe client réutilisable
    """
    
    class MistralClient:
        def __init__(self, tokenizer, model, device):
            self.tokenizer = tokenizer
            self.model = model
            self.device = device
        
        def invoke(self, prompt, max_tokens=200, temperature=0.7):
            """Génère une réponse"""
            # Formater au format Mistral
            if not prompt.startswith("<s>[INST]"):
                formatted_prompt = f"<s>[INST] {prompt} [/INST]"
            else:
                formatted_prompt = prompt
            
            inputs = self.tokenizer(
                formatted_prompt, 
                return_tensors="pt",
                truncation=True,
                max_length=4096
            ).to(self.device)
            
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    top_p=0.9,
                    do_sample=temperature > 0,
                    repetition_penalty=1.1,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            response = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], 
                skip_special_tokens=True
            )
            
            return response.strip()
        
        def __call__(self, prompt, **kwargs):
            """Permet d'appeler directement l'instance"""
            return self.invoke(prompt, **kwargs)
    
    return MistralClient(tokenizer, model, device)

def main():
    """
    Script principal
    """
    print("\n" + "="*70)
    print("🎯 MISTRAL-7B - CHARGEMENT SIMPLIFIÉ")
    print("="*70)
    
    # Charger le modèle
    tokenizer, model, device = load_mistral()
    
    if tokenizer is None or model is None:
        print("\n❌ Échec du chargement")
        return None
    
    # Test d'inférence
    test_inference(tokenizer, model, device)
    
    # Créer le client
    print("\n" + "="*60)
    print("🎁 CRÉATION DU CLIENT")
    print("="*60)
    
    client = create_client_class(tokenizer, model, device)
    
    print("\n✅ Client créé! Exemples d'utilisation:")
    print("""
# Utilisation simple
response = client("Qu'est-ce que Python ?")
print(response)

# Avec paramètres
response = client(
    "Explique la photosynthèse en 3 phrases",
    max_tokens=150,
    temperature=0.5
)
print(response)
    """)
    
    # Test interactif
    print("\n" + "="*60)
    print("💬 TEST INTERACTIF")
    print("="*60)
    
    questions = [
        "Quelle est la capitale de l'Allemagne ?",
        "Cite 3 langages de programmation populaires",
        "Qu'est-ce qu'un transformeur en IA ?"
    ]
    
    for i, question in enumerate(questions, 1):
        print(f"\n❓ Question {i}: {question}")
        response = client(question, max_tokens=100)
        print(f"💬 Réponse: {response}")
        print("-" * 60)
    
    print("\n" + "="*60)
    print("✨ TOUT EST PRÊT!")
    print("="*60)
    
    return client

if __name__ == "__main__":
    client = main()
    
    # Sauvegarder le client globalement pour réutilisation
    if client:
        print("\n💾 Client sauvegardé dans la variable 'client'")
        print("   Utilisez: client('votre question')")

In [ ]:
import os

local_dir = "/home/user/models/mistral-7b"

if os.path.exists(local_dir) and os.listdir(local_dir):
    print("✅ Le modèle est présent sur la machine")
    print("📂 Contenu du dossier :", os.listdir(local_dir)[:10], "...")  # Affiche les 10 premiers fichiers
else:
    print("❌ Le modèle n'est pas trouvé dans le dossier local")


In [ ]:
from vector_store import get_vector_store, search_documents

# Méthode 1: Instance singleton
store = get_vector_store()
scores, indices, documents = store.search("protection différentielle", k=5)

# Méthode 2: Fonction utilitaire
scores, indices, documents = search_documents("mise à la terre", k=3)

# Accès aux résultats
for doc in documents:
    print(f"Score: {doc['similarity_score']:.3f}")
    print(f"Titre: {doc.get('title', 'Sans titre')}")
    print(f"Contenu: {doc.get('content', '')[:100]}...")

In [ ]:
#!/usr/bin/env python3
"""
Test FINAL avec formats EXACTS pour l'API
"""

import requests
import json
import time

BASE_URL = "http://localhost:8000"

def print_test(name, success, response=None):
    """Affiche un résultat de test"""
    icon = "✅" if success else "❌"
    print(f"{icon} {name}")
    if response and response.text:
        try:
            data = response.json()
            print(f"   Réponse: {json.dumps(data, indent=2, ensure_ascii=False)[:150]}...")
        except:
            print(f"   Réponse: {response.text[:150]}...")

def main():
    print("="*80)
    print("TEST DÉFINITIF - FORMATS EXACTS")
    print("="*80)
    
    all_success = True
    
    # 1. Health (GET)
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=5)
        print_test("1. Health", r.status_code == 200, r)
        all_success &= r.status_code == 200
    except Exception as e:
        print_test("1. Health", False)
        print(f"   Erreur: {e}")
        all_success = False
    
    # 2. Status (GET)
    try:
        r = requests.get(f"{BASE_URL}/api/v1/status", timeout=5)
        print_test("2. Status", r.status_code == 200, r)
        all_success &= r.status_code == 200
    except Exception as e:
        print_test("2. Status", False)
        print(f"   Erreur: {e}")
        all_success = False
    
    # 3. AUTOMATIC DISCOVERY - Trouver le bon format pour autocomplete
    print("\n🔍 Découverte du format pour /autocomplete:")
    
    test_formats = [
        {"query": "protéger les"},  # Le plus probable
        {"text": "protéger les"},
        {"input": "protéger les"},
        {"search": "protéger les"},
        {"prefix": "protéger les"},
        {"suggestion": "protéger les"},
    ]
    
    found_format = None
    for i, test_data in enumerate(test_formats, 1):
        try:
            r = requests.post(
                f"{BASE_URL}/api/v1/autocomplete",
                json=test_data,
                timeout=5
            )
            print(f"   Essai {i}: {test_data} -> {r.status_code}")
            if r.status_code == 200:
                found_format = test_data
                print(f"   🎉 Format trouvé: {test_data}")
                break
        except:
            continue
    
    if found_format:
        print_test("3. Autocomplete", True)
    else:
        print_test("3. Autocomplete", False)
        print(f"   ❌ Aucun format ne fonctionne")
        all_success = False
    
    # 4. Extract Norme (POST avec "observation")
    try:
        r = requests.post(
            f"{BASE_URL}/api/v1/extract_norme",
            json={"observation": "Protéger les circuits par un disjoncteur différentiel 30mA"},
            timeout=5
        )
        print_test("4. Extract Norme", r.status_code == 200, r)
        all_success &= r.status_code == 200
    except Exception as e:
        print_test("4. Extract Norme", False)
        print(f"   Erreur: {e}")
        all_success = False
    
    # 5. Découverte du format pour reformulate
    print("\n🔍 Découverte du format pour /reformulate:")
    
    test_formats_reform = [
        {"text": "câble abîmé"},  # Le plus probable d'après l'erreur
        {"observation": "câble abîmé"},
        {"input": "câble abîmé"},
        {"query": "câble abîmé"},
        {"sentence": "câble abîmé"},
    ]
    
    found_format_reform = None
    for i, test_data in enumerate(test_formats_reform, 1):
        try:
            r = requests.post(
                f"{BASE_URL}/api/v1/reformulate",
                json=test_data,
                timeout=5
            )
            print(f"   Essai {i}: {test_data} -> {r.status_code}")
            if r.status_code == 200:
                found_format_reform = test_data
                print(f"   🎉 Format trouvé: {test_data}")
                break
        except:
            continue
    
    if found_format_reform:
        print_test("5. Reformulate", True)
    else:
        print_test("5. Reformulate", False)
        all_success = False
    
    # 6. Search (POST avec "query" et "k")
    try:
        r = requests.post(
            f"{BASE_URL}/api/v1/search",
            json={"query": "protection différentielle 30mA", "k": 5},
            timeout=5
        )
        print_test("6. Search", r.status_code == 200, r)
        all_success &= r.status_code == 200
    except Exception as e:
        print_test("6. Search", False)
        print(f"   Erreur: {e}")
        all_success = False
    
    # 7. API Health (GET)
    try:
        r = requests.get(f"{BASE_URL}/api/v1/health", timeout=5)
        print_test("7. API Health", r.status_code == 200, r)
        all_success &= r.status_code == 200
    except Exception as e:
        print_test("7. API Health", False)
        print(f"   Erreur: {e}")
        all_success = False
    
    print("\n" + "="*80)
    if all_success:
        print("🎉 TOUS LES TESTS PASSENT !")
    else:
        print("⚠️  Certains tests ont échoué")
    print("="*80)
    
    # Afficher les formats trouvés
    print("\n📋 FORMETS DÉFINITIFS :")
    print("1. /health            : GET (pas de body)")
    print("2. /api/v1/status     : GET (pas de body)")
    print("3. /api/v1/autocomplete : POST avec {'query': '...'}")
    print("4. /api/v1/extract_norme : POST avec {'observation': '...'}")
    print("5. /api/v1/reformulate  : POST avec {'text': '...'}")
    print("6. /api/v1/search     : POST avec {'query': '...', 'k': 5}")
    print("7. /api/v1/health     : GET (pas de body)")

if __name__ == "__main__":
    main()

In [ ]:
"""
Configuration centralisée pour le projet RAG - Normes électriques
Compatible scripts Python ET Jupyter Notebook
"""

from pathlib import Path
import logging
import sys


# =============================================================================
# DÉTECTION AUTOMATIQUE DU RÉPERTOIRE RACINE (My_project)
# =============================================================================

def detect_base_dir():
    """
    Détecte automatiquement le répertoire racine du projet (My_project)
    Fonctionne depuis n'importe quel sous-dossier ou notebook
    """
    # Méthode 1 : depuis __file__ (fonctionne pour les scripts)
    try:
        current_file = Path(__file__).resolve()
        # Remonter jusqu'à trouver My_project
        for parent in [current_file.parent] + list(current_file.parents):
            # D'abord chercher par nom exact
            if parent.name == "My_project":
                return parent
            # Puis chercher un marqueur fiable (data/ avec index.faiss)
            if (parent / "data" / "index.faiss").exists():
                return parent
    except NameError:
        pass  # __file__ non défini (Jupyter)
    
    # Méthode 2 : depuis le répertoire courant (Jupyter/script)
    current = Path.cwd()
    
    # Cas 1 : on est dans My_project/ directement
    if (current / "data" / "index.faiss").exists():
        return current
    
    # Cas 2 : on est dans un sous-dossier (ex: My_project/script/)
    for parent in [current] + list(current.parents):
        if parent.name == "My_project":
            return parent
        if (parent / "data" / "index.faiss").exists():
            return parent
    
    # Cas 3 : on est au-dessus de My_project/ (ex: PROJET RAPPORTS/)
    if (current / "My_project" / "data" / "index.faiss").exists():
        return current / "My_project"
    
    # Méthode 3 : variable d'environnement (optionnel)
    import os
    if "MY_PROJECT_ROOT" in os.environ:
        return Path(os.environ["MY_PROJECT_ROOT"])
    
    # Par défaut : répertoire courant
    fallback = Path.cwd()
    print(f"⚠️  Impossible de détecter My_project, utilisation de : {fallback}")
    print(f"⚠️  Recherché : {fallback}/data/index.faiss")
    return fallback


# =============================================================================
# CHEMINS VALIDÉS
# =============================================================================

BASE_DIR = detect_base_dir()

DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
SCRIPT_DIR = BASE_DIR / "script"

# Fichiers FAISS existants dans ton dossier
FAISS_INDEX_PATH = DATA_DIR / "index.faiss"
METADATA_PATH = DATA_DIR / "index.pkl"

DOCUMENTS_JSON_DIR = DATA_DIR / "document_json"
NORMES_DIR = DATA_DIR / "normes"

# Créer les répertoires s'ils n'existent pas
for d in [DATA_DIR, MODELS_DIR, SCRIPT_DIR, DOCUMENTS_JSON_DIR, NORMES_DIR]:
    d.mkdir(exist_ok=True, parents=True)


# =============================================================================
# MODÈLES
# =============================================================================

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSION = 384
LLM_MODEL = "llama-3.1-70b-versatile"


# =============================================================================
# PARAMÈTRES RAG
# =============================================================================

TOP_K = 5
MAX_CONTEXT_TOKENS = 2000
SIMILARITY_THRESHOLD = 0.7
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50


# =============================================================================
# FICHIERS SUPPORTÉS
# =============================================================================

SUPPORTED_EXTENSIONS = {".txt", ".pdf", ".json", ".doc", ".docx", ".md"}
ENCODINGS = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]


# =============================================================================
# LOGGING
# =============================================================================

LOGGING_CONFIG = {
    "level": logging.INFO,
    "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
}

logging.basicConfig(**LOGGING_CONFIG)
logger = logging.getLogger(__name__)


# =============================================================================
# PROMPTS
# =============================================================================

SYSTEM_PROMPT = """Tu es un expert en normes électriques françaises (NF C 15-100).
Tes réponses sont techniques, précises et justifiées."""

RAG_PROMPT_TEMPLATE = """
CONTEXTE:
{context}

QUESTION:
{question}

Réponds de manière structurée et cite les références normatives pertinentes.
"""


# =============================================================================
# VALIDATION CONFIG
# =============================================================================

def valider_configuration():
    """Valide la configuration et affiche les avertissements/erreurs"""
    errors = []
    warnings = []

    # Vérification du répertoire de données
    if not DATA_DIR.exists():
        errors.append(f"❌ DATA_DIR introuvable : {DATA_DIR}")
    else:
        logger.info(f"✅ DATA_DIR trouvé : {DATA_DIR}")

    # Vérification des paramètres de chunking
    if CHUNK_SIZE <= CHUNK_OVERLAP:
        errors.append("❌ CHUNK_SIZE doit être > CHUNK_OVERLAP")

    # Vérification de l'index FAISS
    if not FAISS_INDEX_PATH.exists():
        warnings.append(f"⚠️  Index FAISS introuvable : {FAISS_INDEX_PATH}")
    else:
        logger.info(f"✅ Index FAISS trouvé : {FAISS_INDEX_PATH}")

    # Vérification des métadonnées
    if not METADATA_PATH.exists():
        warnings.append(f"⚠️  Métadonnées introuvables : {METADATA_PATH}")
    else:
        logger.info(f"✅ Métadonnées trouvées : {METADATA_PATH}")

    # Afficher les avertissements
    for w in warnings:
        logger.warning(w)

    # Lever une erreur si problèmes critiques
    if errors:
        error_msg = "\n".join(errors)
        logger.error(f"Configuration invalide:\n{error_msg}")
        raise ValueError(error_msg)

    logger.info("✅ Configuration validée avec succès")
    return True


# =============================================================================
# AFFICHAGE DE LA CONFIGURATION (lors de l'import)
# =============================================================================

def print_config_summary():
    """Affiche un résumé de la configuration"""
    print("=" * 60)
    print("📋 CONFIGURATION DU PROJET RAG")
    print("=" * 60)
    print(f"🏠 BASE_DIR       : {BASE_DIR}")
    print(f"📁 DATA_DIR       : {DATA_DIR}")
    print(f"📁 MODELS_DIR     : {MODELS_DIR}")
    print(f"📁 SCRIPT_DIR     : {SCRIPT_DIR}")
    print(f"📊 FAISS_INDEX    : {FAISS_INDEX_PATH}")
    print(f"📝 METADATA       : {METADATA_PATH}")
    print(f"🤖 EMBEDDING_MODEL: {EMBEDDING_MODEL}")
    print(f"🧠 LLM_MODEL      : {LLM_MODEL}")
    print("=" * 60)
    
    # État des fichiers critiques
    print("\n📂 État des fichiers:")
    print(f"  {'✅' if FAISS_INDEX_PATH.exists() else '❌'} Index FAISS")
    print(f"  {'✅' if METADATA_PATH.exists() else '❌'} Métadonnées")
    print(f"  {'✅' if DATA_DIR.exists() else '❌'} Répertoire data/")
    print("=" * 60 + "\n")


# Afficher le résumé lors de l'import (optionnel, commenter si gênant)
# print_config_summary()


# =============================================================================
# EXPORT (important !)
# =============================================================================

__all__ = [
    "BASE_DIR", "DATA_DIR", "MODELS_DIR", "SCRIPT_DIR",
    "FAISS_INDEX_PATH", "METADATA_PATH",
    "DOCUMENTS_JSON_DIR", "NORMES_DIR",
    "EMBEDDING_MODEL", "EMBEDDING_DIMENSION", "LLM_MODEL",
    "TOP_K", "MAX_CONTEXT_TOKENS", "SIMILARITY_THRESHOLD",
    "CHUNK_SIZE", "CHUNK_OVERLAP",
    "SUPPORTED_EXTENSIONS", "ENCODINGS",
    "LOGGING_CONFIG",
    "SYSTEM_PROMPT", "RAG_PROMPT_TEMPLATE",
    "valider_configuration",
    "detect_base_dir",
    "print_config_summary"
]

In [ ]:
# test_final.py
import sys
from pathlib import Path

print("🧪 TEST FINAL DES CHEMINS")
print("=" * 50)

# Test 1: Où sommes-nous vraiment ?
current_dir = Path.cwd()
print(f"📁 Dossier courant: {current_dir}")
print(f"📁 Nom: {current_dir.name}")

# Test 2: Contenu du dossier courant
print(f"\n📂 Contenu de {current_dir}:")
for item in current_dir.iterdir():
    type_str = "📁" if item.is_dir() else "📄"
    print(f"   {type_str} {item.name}")

# Test 3: Vérifier data/ depuis ici
data_dir = current_dir / "data"
print(f"\n🔍 Vérification data/: {data_dir}")
print(f"   Existe: {data_dir.exists()}")

if data_dir.exists():
    print(f"   Contenu: {list(data_dir.glob('*'))}")

# Test 4: Import config
print(f"\n🔄 Test import config...")
try:
    sys.path.append(str(current_dir / "script"))
    from config import BASE_DIR, DATA_DIR, FAISS_INDEX_PATH
    
    print(f"✅ Import réussi")
    print(f"📁 BASE_DIR: {BASE_DIR}")
    print(f"📁 DATA_DIR: {DATA_DIR}")
    print(f"📄 FAISS: {FAISS_INDEX_PATH.exists()}")
    
except Exception as e:
    print(f"❌ Erreur import: {e}")

In [ ]:
"""
Pipeline RAG complet pour l'analyse d'observations électriques
Intégration Retriever + ContextBuilder + LLM pour génération de réponses expertes
"""

import logging
import json
from typing import Dict, Any, Optional, List
from dataclasses import dataclass
from config import (
    SYSTEM_PROMPT,
    RAG_PROMPT_TEMPLATE,
    #OBSERVATION_PROMPT,
    LLM_MODEL,
    TOP_K
)

# Import conditionnel pour OpenAI
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI non disponible. Installation: pip install openai")

from retriever import get_retriever, retrieve_for_observation
from context_builder import get_context_builder, build_rag_context

# Configuration du logging
logger = logging.getLogger(__name__)

@dataclass
class PipelineResponse:
    """Réponse structurée du pipeline RAG"""
    observation_corrigee: str
    terminologie_enrichie: List[str]
    normes_applicables: List[str]
    analyse_technique: str
    recommandations: List[str]
    contexte_utilise: Dict[str, Any]
    metadata: Dict[str, Any]

class RAGPipeline:
    """
    Pipeline RAG complet pour l'analyse d'observations électriques
    """
    
    def __init__(self, retriever=None, context_builder=None, llm_model: str = None):
        """
        Initialise le pipeline RAG
        
        Args:
            retriever: Instance de Retriever
            context_builder: Instance de ContextBuilder
            llm_model: Modèle LLM à utiliser
        """
        self.retriever = retriever or get_retriever()
        self.context_builder = context_builder or get_context_builder()
        self.llm_model = llm_model or LLM_MODEL
        self.llm_client = None
        
        self._initialize_llm()
        logger.info(f"✅ Pipeline RAG initialisé avec modèle: {self.llm_model}")
    
    def _initialize_llm(self) -> None:
        """Initialise le client LLM"""
        if self.llm_model.startswith("gpt-"):
            if not OPENAI_AVAILABLE:
                raise ImportError("OpenAI non disponible. pip install openai")
            # Le client OpenAI sera initialisé avec la clé API de l'utilisateur
            self.llm_client = "openai"
        else:
            # Support pour d'autres modèles (LLaMA, Mistral, etc.)
            self.llm_client = "custom"
            logger.warning("⚠️  Modèle personnalisé - implémentez votre client LLM")
    
    def process_observation(self, observation: str, k: int = None) -> PipelineResponse:
        """
        Traite une observation électrique complète
        
        Args:
            observation: Observation technique brute
            k: Nombre de documents à récupérer
            
        Returns:
            Réponse structurée du pipeline
        """
        try:
            logger.info(f"🔧 Traitement observation: '{observation[:50]}...'")
            
            # Étape 1: Recherche des documents pertinents
            retrieval_results = self.retriever.retrieve_for_observation(observation, k)
            
            # Étape 2: Construction du contexte RAG
            context_result = self.context_builder.build_context(retrieval_results)
            
            # Étape 3: Génération de la réponse par LLM
            llm_response = self._generate_llm_response(observation, context_result)
            
            # Étape 4: Structuration de la réponse finale
            final_response = self._structure_final_response(
                observation, llm_response, context_result, retrieval_results
            )
            
            logger.info("✅ Traitement observation terminé avec succès")
            return final_response
            
        except Exception as e:
            logger.error(f"❌ Erreur traitement observation: {e}")
            return self._build_error_response(observation, str(e))
    
    def _generate_llm_response(self, observation: str, context_result: Dict[str, Any]) -> Dict[str, Any]:
        """
        Génère une réponse via le LLM
        """
        if not context_result["metadata"]["is_valid"]:
            return self._generate_fallback_response(observation)
        
        try:
            # Construction du prompt optimisé
            prompt = self._build_optimized_prompt(observation, context_result)
            
            # Appel au LLM
            if self.llm_client == "openai":
                llm_output = self._call_openai(prompt)
            else:
                llm_output = self._call_custom_llm(prompt)
            
            # Parsing de la réponse
            parsed_response = self._parse_llm_response(llm_output)
            
            return parsed_response
            
        except Exception as e:
            logger.error(f"❌ Erreur génération LLM: {e}")
            return self._generate_fallback_response(observation)
    
    def _build_optimized_prompt(self, observation: str, context_result: Dict[str, Any]) -> str:
        """
        Construit un prompt optimisé pour l'analyse électrique
        """
        context_text = context_result["context_text"]
        
        prompt = f"""
{SYSTEM_PROMPT}

{OBSERVATION_PROMPT.format(observation=observation, context=context_text)}

Votre analyse doit structurer la réponse en respectant STRICTEMENT le format JSON suivant :

{{
    "observation_corrigee": "Observation reformulée avec terminologie technique exacte",
    "terminologie_enrichie": ["liste", "des", "termes", "techniques", "pertinents"],
    "normes_applicables": ["NF C 15-100 § XXX", "NF C 15-100 § YYY"],
    "analyse_technique": "Analyse détaillée du problème et des risques",
    "recommandations": ["Action corrective 1", "Action corrective 2"]
}}

Exigences supplémentaires :
- Utilisez UNIQUEMENT le contexte fourni pour les références normatives
- Soyez technique et précis
- Proposez des actions concrètes et réalisables
- Mentionnez les risques de non-conformité

RÉPONSE (format JSON uniquement):
"""
        return prompt
    
    def _call_openai(self, prompt: str) -> str:
        """
        Appelle l'API OpenAI
        """
        # Note: L'utilisateur doit configurer sa clé API OpenAI
        # openai.api_key = "votre_cle_api"
        
        try:
            response = openai.ChatCompletion.create(
                model=self.llm_model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,  # Faible température pour des réponses stables
                max_tokens=1500,
                response_format={"type": "json_object"}  # Forcer le format JSON
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            logger.error(f"❌ Erreur appel OpenAI: {e}")
            raise
    
    def _call_custom_llm(self, prompt: str) -> str:
        """
        Appelle un modèle LLM personnalisé
        """
        # Implémentation à adapter selon le modèle utilisé
        # Exemple pour LLaMA, Mistral, etc.
        logger.warning("⚠️  Implémentez votre client LLM personnalisé")
        
        # Simulation de réponse pour les tests
        return json.dumps({
            "observation_corrigee": "Absence de disjoncteur différentiel 30mA de type A",
            "terminologie_enrichie": ["disjoncteur différentiel", "sensibilité 30mA", "type A", "protection personnes"],
            "normes_applicables": ["NF C 15-100 § 415.1", "NF C 15-100 § 531.2"],
            "analyse_technique": "L'absence de protection différentielle expose les personnes aux risques de électrocution par contact indirect. La norme exige un dispositif différentiel pour tous les circuits terminaux.",
            "recommandations": [
                "Installer un disjoncteur différentiel 30mA type A en tête des circuits concernés",
                "Vérifier la continuité des conducteurs de protection",
                "Contrôler la valeur de la prise de terre"
            ]
        })
    
    def _parse_llm_response(self, llm_output: str) -> Dict[str, Any]:
        """
        Parse la réponse du LLM en structure de données
        """
        try:
            # Nettoyer la réponse
            clean_output = llm_output.strip()
            if clean_output.startswith("```json"):
                clean_output = clean_output[7:]
            if clean_output.endswith("```"):
                clean_output = clean_output[:-3]
            
            # Parsing JSON
            parsed = json.loads(clean_output)
            
            # Validation des champs requis
            required_fields = [
                "observation_corrigee", "terminologie_enrichie", 
                "normes_applicables", "analyse_technique", "recommandations"
            ]
            
            for field in required_fields:
                if field not in parsed:
                    parsed[field] = "" if field != "recommandations" else []
            
            return parsed
            
        except json.JSONDecodeError as e:
            logger.error(f"❌ Erreur parsing JSON LLM: {e}")
            logger.debug(f"Contenu LLM: {llm_output}")
            return self._generate_fallback_parsing(llm_output)
    
    def _generate_fallback_response(self, observation: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours si le contexte est invalide
        """
        return {
            "observation_corrigee": observation,
            "terminologie_enrichie": ["observation à analyser"],
            "normes_applicables": ["NF C 15-100 (référence générale)"],
            "analyse_technique": "Impossible d'analyser cette observation avec les données disponibles. Vérifiez la formulation ou consultez un expert.",
            "recommandations": [
                "Reformuler l'observation avec plus de détails techniques",
                "Consulter la documentation normative NF C 15-100",
                "Contacter un organisme de certification"
            ]
        }
    
    def _generate_fallback_parsing(self, llm_output: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours en cas d'erreur de parsing
        """
        return {
            "observation_corrigee": "Analyse nécessitant validation manuelle",
            "terminologie_enrichie": ["validation expert requise"],
            "normes_applicables": ["NF C 15-100 § applicable"],
            "analyse_technique": f"Réponse LLM nécessitant interprétation: {llm_output[:200]}...",
            "recommandations": [
                "Faire vérifier l'analyse par un expert qualifié",
                "Consulter les documents normatifs directement"
            ]
        }
    
    def _structure_final_response(self, original_observation: str, llm_response: Dict[str, Any],
                                context_result: Dict[str, Any], retrieval_results: Dict[str, Any]) -> PipelineResponse:
        """
        Structure la réponse finale du pipeline
        """
        # Métadonnées de traitement
        metadata = {
            "contexte_valide": context_result["metadata"]["is_valid"],
            "documents_utilises": context_result["metadata"]["total_sections"],
            "types_documents": context_result["metadata"]["document_types"],
            "score_moyen": context_result["metadata"]["average_score"],
            "modele_llm": self.llm_model,
            "categorie_detectee": retrieval_results.get("categorie_detectee", "inconnue")
        }
        
        # Informations de contexte utilisées
        contexte_utilise = {
            "sections_count": len(context_result["sections_details"]),
            "sources": list(set(section["source"] for section in context_result["sections_details"])),
            "contexte_length": context_result["metadata"]["context_length"]
        }
        
        return PipelineResponse(
            observation_corrigee=llm_response["observation_corrigee"],
            terminologie_enrichie=llm_response["terminologie_enrichie"],
            normes_applicables=llm_response["normes_applicables"],
            analyse_technique=llm_response["analyse_technique"],
            recommandations=llm_response["recommandations"],
            contexte_utilise=contexte_utilise,
            metadata=metadata
        )
    
    def _build_error_response(self, observation: str, error_msg: str) -> PipelineResponse:
        """
        Construit une réponse d'erreur
        """
        return PipelineResponse(
            observation_corrigee=observation,
            terminologie_enrichie=["erreur traitement"],
            normes_applicables=[],
            analyse_technique=f"Erreur lors du traitement: {error_msg}",
            recommandations=["Vérifier la configuration du pipeline", "Contacter le support technique"],
            contexte_utilise={},
            metadata={"erreur": error_msg, "contexte_valide": False}
        )
    
    def generate_report(self, response: PipelineResponse, format: str = "text") -> str:
        """
        Génère un rapport formaté à partir de la réponse
        
        Args:
            response: Réponse du pipeline
            format: Format de sortie ('text', 'markdown', 'json')
            
        Returns:
            Rapport formaté
        """
        if format == "json":
            return json.dumps({
                "observation_corrigee": response.observation_corrigee,
                "terminologie_enrichie": response.terminologie_enrichie,
                "normes_applicables": response.normes_applicables,
                "analyse_technique": response.analyse_technique,
                "recommandations": response.recommandations,
                "metadata": response.metadata
            }, indent=2, ensure_ascii=False)
        
        elif format == "markdown":
            return self._generate_markdown_report(response)
        
        else:  # format text
            return self._generate_text_report(response)
    
    def _generate_markdown_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format Markdown"""
        report = [
            "# RAPPORT D'ANALYSE ÉLECTRIQUE",
            "",
            f"## Observation corrigée",
            f"{response.observation_corrigee}",
            "",
            f"## Terminologie technique",
            "\n".join(f"- {term}" for term in response.terminologie_enrichie),
            "",
            f"## Normes applicables",
            "\n".join(f"- {norme}" for norme in response.normes_applicables),
            "",
            f"## Analyse technique",
            f"{response.analyse_technique}",
            "",
            f"## Recommandations",
            "\n".join(f"{i+1}. {reco}" for i, reco in enumerate(response.recommandations)),
            "",
            f"## Métadonnées",
            f"- Contexte valide: {response.metadata.get('contexte_valide', 'N/A')}",
            f"- Documents utilisés: {response.metadata.get('documents_utilises', 'N/A')}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ]
        
        return "\n".join(report)
    
    def _generate_text_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format texte simple"""
        report = [
            "RAPPORT D'ANALYSE ÉLECTRIQUE",
            "=" * 50,
            "",
            f"OBSERVATION CORRIGÉE:",
            f"{response.observation_corrigee}",
            "",
            f"TERMINOLOGIE TECHNIQUE:",
            f"{', '.join(response.terminologie_enrichie)}",
            "",
            f"NORMES APPLICABLES:",
            f"{', '.join(response.normes_applicables)}",
            "",
            f"ANALYSE TECHNIQUE:",
            f"{response.analyse_technique}",
            "",
            f"RECOMMANDATIONS:",
        ]
        
        for i, reco in enumerate(response.recommandations, 1):
            report.append(f"{i}. {reco}")
        
        report.extend([
            "",
            f"INFORMATIONS:",
            f"- Sources utilisées: {len(response.contexte_utilise.get('sources', []))}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ])
        
        return "\n".join(report)


# Instance singleton pour une utilisation facile
_pipeline_instance = None

def get_pipeline() -> RAGPipeline:
    """
    Retourne une instance singleton du pipeline RAG
    
    Returns:
        Instance de RAGPipeline
    """
    global _pipeline_instance
    
    if _pipeline_instance is None:
        _pipeline_instance = RAGPipeline()
    
    return _pipeline_instance

def process_electrical_observation(observation: str, k: int = None) -> PipelineResponse:
    """
    Fonction utilitaire pour traitement rapide d'observation
    
    Args:
        observation: Observation technique
        k: Nombre de documents
        
    Returns:
        Réponse structurée
    """
    pipeline = get_pipeline()
    return pipeline.process_observation(observation, k)

# Test du module
if __name__ == "__main__":
    print("🧪 Test du Pipeline RAG...")
    
    try:
        # Configuration pour test (sans appel réel à OpenAI)
        import os
        os.environ["OPENAI_API_KEY"] = "test-key"  # Clé factice pour test
        
        pipeline = RAGPipeline(llm_model="gpt-3.5-turbo")
        
        # Test avec une observation électrique
        test_observation = "Disjoncteur différentiel 30mA manquant dans le tableau cuisine"
        response = pipeline.process_observation(test_observation)
        
        print("✅ Pipeline RAG testé avec succès")
        print(f"📊 Observation corrigée: {response.observation_corrigee}")
        print(f"🔧 Normes applicables: {response.normes_applicables}")
        print(f"📝 Recommandations: {len(response.recommandations)}")
        
        # Génération de rapport
        report = pipeline.generate_report(response, "text")
        print(f"\n📄 Rapport généré ({len(report)} caractères)")
        
        print("✅ Test passé avec succès")
        
    except Exception as e:
        print(f"❌ Test échoué: {e}")

In [ ]:
"""
Module ContextBuilder pour le pipeline RAG - Construction du contexte final
Assemblage et validation des résultats de recherche pour alimenter le modèle
"""

import logging
from typing import List, Dict, Any, Optional, Set
from dataclasses import dataclass
from config import (
    MAX_CONTEXT_TOKENS,
    SYSTEM_PROMPT,
    RAG_PROMPT_TEMPLATE
)

# Configuration du logging
logger = logging.getLogger(__name__)

@dataclass
class DocumentSection:
    """Section de document normalisée"""
    content: str
    source: str
    doc_type: str
    score: float
    metadata: Dict[str, Any]

class ContextBuilder:
    """
    Constructeur de contexte pour le pipeline RAG
    Assemble, valide et structure les résultats de recherche
    """
    
    def __init__(self, max_tokens: int = None):
        """
        Initialise le ContextBuilder
        
        Args:
            max_tokens: Nombre maximum de tokens pour le contexte
        """
        self.max_tokens = max_tokens or MAX_CONTEXT_TOKENS
        self.estimated_tokens_per_char = 0.25  # Estimation conservatrice
        
        # Priorités des types de documents dans l'assemblage
        self.type_priority = {
            "normes": 4,      # Priorité maximale
            "correspondances": 3,
            "principaux": 2,
            "definitions": 1,
            "default": 0
        }
        
        # Configuration de l'assemblage
        self.section_config = {
            "max_section_length": 500,  # Caractères par section
            "min_content_length": 10,   # Contenu minimum
            "separator": "\n\n---\n\n"   # Séparateur entre sections
        }
        
        logger.info("✅ ContextBuilder initialisé")
    
    def build_context(self, retrieval_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Construit le contexte final à partir des résultats de recherche
        
        Args:
            retrieval_results: Résultats du Retriever
            
        Returns:
            Contexte structuré pour le modèle
        """
        try:
            # Extraction et validation des documents
            all_documents = self._extract_and_validate_documents(retrieval_results)
            
            if not all_documents:
                logger.warning("⚠️  Aucun document valide trouvé")
                return self._build_empty_context(retrieval_results)
            
            # Assemblage intelligent des sections
            context_sections = self._assemble_context_sections(all_documents)
            
            # Construction du texte de contexte
            context_text = self._build_context_text(context_sections)
            
            # Validation finale du contexte
            is_valid = self._validate_final_context(context_text, context_sections)
            
            # Construction de la réponse structurée
            final_context = self._structure_final_context(
                context_text, context_sections, retrieval_results, is_valid
            )
            
            logger.info(f"📚 Contexte construit: {len(context_sections)} sections, "
                       f"{len(context_text)} caractères, valid: {is_valid}")
            
            return final_context
            
        except Exception as e:
            logger.error(f"❌ Erreur construction contexte: {e}")
            return self._build_error_context(retrieval_results, str(e))
    
    def _extract_and_validate_documents(self, retrieval_results: Dict) -> List[DocumentSection]:
        """
        Extrait et valide les documents des résultats de recherche
        """
        documents = []
        seen_content = set()  # Pour éviter les doublons
        
        # Parcourir toutes les catégories de résultats
        for category, doc_list in retrieval_results.get("resultats", {}).items():
            if not isinstance(doc_list, list):
                continue
                
            for doc in doc_list:
                try:
                    # Validation de base
                    if not self._is_valid_document(doc):
                        continue
                    
                    # Nettoyer et normaliser le contenu
                    clean_content = self._clean_document_content(
                        doc.get('content', ''),
                        doc.get('title', '')
                    )
                    
                    # Vérifier les doublons
                    content_hash = hash(clean_content[:100])  # Hash du début pour détection doublons
                    if content_hash in seen_content:
                        continue
                    seen_content.add(content_hash)
                    
                    # Créer la section document
                    section = DocumentSection(
                        content=clean_content,
                        source=doc.get('source', 'Unknown'),
                        doc_type=category,
                        score=doc.get('weighted_score', doc.get('similarity_score', 0)),
                        metadata={
                            'title': doc.get('title', ''),
                            'category': category,
                            'original_score': doc.get('similarity_score', 0),
                            'weighted_score': doc.get('weighted_score', 0)
                        }
                    )
                    
                    documents.append(section)
                    
                except Exception as e:
                    logger.warning(f"⚠️  Document ignoré (erreur traitement): {e}")
                    continue
        
        # Trier par score pondéré
        documents.sort(key=lambda x: x.score, reverse=True)
        
        logger.debug(f"📄 Documents validés: {len(documents)} sur {sum(len(docs) for docs in retrieval_results.get('resultats', {}).values())} initiaux")
        return documents
    
    def _is_valid_document(self, doc: Dict) -> bool:
        """
        Valide qu'un document est utilisable
        """
        # Vérifier présence contenu
        content = doc.get('content', '').strip()
        if not content or len(content) < self.section_config["min_content_length"]:
            return False
        
        # Vérifier que ce n'est pas un document d'erreur
        if any(error_indicator in content.lower() for error_indicator in ['erreur', 'error', 'not found', 'introuvable']):
            return False
        
        return True
    
    def _clean_document_content(self, content: str, title: str = "") -> str:
        """
        Nettoie et formate le contenu d'un document
        """
        # Nettoyer le contenu
        clean_content = content.strip()
        
        # Ajouter le titre si pertinent
        if title and len(title) < 100 and title.lower() not in content.lower()[:200]:
            clean_content = f"# {title}\n\n{clean_content}"
        
        # Tronquer si trop long
        max_len = self.section_config["max_section_length"]
        if len(clean_content) > max_len:
            clean_content = clean_content[:max_len] + "..."
        
        return clean_content
    
    def _assemble_context_sections(self, documents: List[DocumentSection]) -> List[DocumentSection]:
        """
        Assemble intelligemment les sections de contexte
        """
        if not documents:
            return []
        
        # Grouper par type et prioriser
        grouped_sections = {}
        for doc in documents:
            doc_type = doc.doc_type
            if doc_type not in grouped_sections:
                grouped_sections[doc_type] = []
            grouped_sections[doc_type].append(doc)
        
        # Ordonner par priorité de type
        ordered_types = sorted(
            grouped_sections.keys(),
            key=lambda x: self.type_priority.get(x, 0),
            reverse=True
        )
        
        # Sélectionner les meilleures sections par type
        selected_sections = []
        remaining_tokens = self.max_tokens
        
        for doc_type in ordered_types:
            sections = grouped_sections[doc_type][:3]  # Max 3 par type
            
            for section in sections:
                section_tokens = self._estimate_tokens(section.content)
                
                if section_tokens <= remaining_tokens:
                    selected_sections.append(section)
                    remaining_tokens -= section_tokens
                else:
                    break
            
            if remaining_tokens <= 0:
                break
        
        # S'assurer d'avoir au moins quelques sections
        if not selected_sections and documents:
            selected_sections = documents[:min(3, len(documents))]
        
        logger.debug(f"🔧 Sections assemblées: {len(selected_sections)} types: {list(set(s.doc_type for s in selected_sections))}")
        return selected_sections
    
    def _build_context_text(self, sections: List[DocumentSection]) -> str:
        """
        Construit le texte de contexte final
        """
        if not sections:
            return "Aucune information contextuelle disponible."
        
        context_parts = []
        
        # Ajouter un en-tête de contexte
        context_parts.append("# CONTEXTE TECHNIQUE\n")
        context_parts.append("Documents de référence pour l'analyse :\n")
        
        # Ajouter les sections
        for i, section in enumerate(sections, 1):
            section_header = f"\n## Document {i} - {section.doc_type.upper()}"
            if section.metadata.get('title'):
                section_header += f" - {section.metadata['title']}"
            
            context_parts.append(section_header)
            context_parts.append(section.content)
            
            # Ajouter le séparateur sauf pour le dernier
            if i < len(sections):
                context_parts.append(self.section_config["separator"])
        
        # Ajouter les métadonnées de contexte
        context_parts.append("\n\n---\n")
        context_parts.append("**Sources utilisées :**\n")
        for i, section in enumerate(sections, 1):
            source_info = f"- Doc {i}: {section.source} (score: {section.score:.3f})"
            context_parts.append(source_info)
        
        context_text = "\n".join(context_parts)
        
        # Tronquer si nécessaire (très rare)
        estimated_tokens = self._estimate_tokens(context_text)
        if estimated_tokens > self.max_tokens:
            context_text = self._truncate_context(context_text)
            logger.warning(f"⚠️  Contexte tronqué: {estimated_tokens} -> {self._estimate_tokens(context_text)} tokens")
        
        return context_text
    
    def _validate_final_context(self, context_text: str, sections: List[DocumentSection]) -> bool:
        """
        Valide le contexte final
        """
        # Vérifier longueur minimale
        if len(context_text.strip()) < 50:
            return False
        
        # Vérifier contenu significatif
        if "Aucune information" in context_text:
            return False
        
        # Vérifier diversité des sources
        unique_sources = len(set(section.source for section in sections))
        if unique_sources < 1:
            return False
        
        # Vérifier absence de sections vides
        for section in sections:
            if not section.content.strip():
                return False
        
        return True
    
    def _structure_final_context(self, context_text: str, sections: List[DocumentSection], 
                               retrieval_results: Dict, is_valid: bool) -> Dict[str, Any]:
        """
        Structure la réponse finale
        """
        # Statistiques
        total_documents = len(sections)
        avg_score = sum(section.score for section in sections) / total_documents if sections else 0
        document_types = list(set(section.doc_type for section in sections))
        
        return {
            "context_text": context_text,
            "metadata": {
                "is_valid": is_valid,
                "total_sections": total_documents,
                "document_types": document_types,
                "average_score": round(avg_score, 3),
                "estimated_tokens": self._estimate_tokens(context_text),
                "context_length": len(context_text)
            },
            "sections_details": [
                {
                    "doc_type": section.doc_type,
                    "source": section.source,
                    "score": round(section.score, 3),
                    "content_preview": section.content[:100] + "..." if len(section.content) > 100 else section.content
                }
                for section in sections
            ],
            "retrieval_info": {
                "original_observation": retrieval_results.get("observation_originale", ""),
                "detected_category": retrieval_results.get("categorie_detectee", ""),
                "total_retrieved_docs": sum(len(docs) for docs in retrieval_results.get("resultats", {}).values())
            }
        }
    
    def _estimate_tokens(self, text: str) -> int:
        """
        Estime le nombre de tokens dans un texte
        """
        return int(len(text) * self.estimated_tokens_per_char)
    
    def _truncate_context(self, context_text: str) -> str:
        """
        Tronque intelligemment le contexte si trop long
        """
        max_chars = int(self.max_tokens / self.estimated_tokens_per_char)
        
        if len(context_text) <= max_chars:
            return context_text
        
        # Trouver le dernier séparateur avant la limite
        separator = self.section_config["separator"]
        last_separator_pos = context_text.rfind(separator, 0, max_chars)
        
        if last_separator_pos != -1:
            return context_text[:last_separator_pos] + "\n\n[Contexte tronqué pour respecter les limites...]"
        else:
            return context_text[:max_chars] + "\n\n[Contexte tronqué...]"
    
    def _build_empty_context(self, retrieval_results: Dict) -> Dict[str, Any]:
        """
        Construit un contexte vide en cas d'échec
        """
        return {
            "context_text": "Aucun document pertinent n'a été trouvé dans la base de connaissances. "
                           "Veuillez reformuler votre demande ou vérifier les données disponibles.",
            "metadata": {
                "is_valid": False,
                "total_sections": 0,
                "document_types": [],
                "average_score": 0,
                "estimated_tokens": 0,
                "context_length": 0
            },
            "sections_details": [],
            "retrieval_info": {
                "original_observation": retrieval_results.get("observation_originale", ""),
                "detected_category": retrieval_results.get("categorie_detectee", ""),
                "total_retrieved_docs": 0
            }
        }
    
    def _build_error_context(self, retrieval_results: Dict, error_msg: str) -> Dict[str, Any]:
        """
        Construit un contexte d'erreur
        """
        return {
            "context_text": f"Erreur lors de la construction du contexte: {error_msg}",
            "metadata": {
                "is_valid": False,
                "total_sections": 0,
                "document_types": [],
                "average_score": 0,
                "estimated_tokens": 0,
                "context_length": 0
            },
            "sections_details": [],
            "retrieval_info": {
                "original_observation": retrieval_results.get("observation_originale", ""),
                "detected_category": retrieval_results.get("categorie_detectee", ""),
                "total_retrieved_docs": 0
            },
            "error": error_msg
        }


# Instance singleton pour une utilisation facile
_context_builder_instance = None

def get_context_builder() -> ContextBuilder:
    """
    Retourne une instance singleton du ContextBuilder
    
    Returns:
        Instance de ContextBuilder
    """
    global _context_builder_instance
    
    if _context_builder_instance is None:
        _context_builder_instance = ContextBuilder()
    
    return _context_builder_instance

def build_rag_context(retrieval_results: Dict[str, Any]) -> Dict[str, Any]:
    """
    Fonction utilitaire pour construction rapide de contexte RAG
    
    Args:
        retrieval_results: Résultats du Retriever
        
    Returns:
        Contexte structuré pour le modèle
    """
    builder = get_context_builder()
    return builder.build_context(retrieval_results)

# Test du module
if __name__ == "__main__":
    print("🧪 Test du ContextBuilder...")
    
    try:
        # Données de test simulées
        test_retrieval_results = {
            "observation_originale": "Disjoncteur différentiel manquant",
            "categorie_detectee": "protection",
            "resultats": {
                "principaux": [
                    {
                        "content": "La norme NF C 15-100 exige la présence d'un disjoncteur différentiel 30mA pour protéger les circuits...",
                        "source": "NF_C15-100_2024.pdf",
                        "title": "Exigences protection différentielle",
                        "similarity_score": 0.89,
                        "weighted_score": 0.92
                    }
                ],
                "normes": [
                    {
                        "content": "Section 531.2: Les dispositifs différentiels doivent avoir une sensibilité de 30mA maximum...",
                        "source": "Norme_531.pdf", 
                        "title": "Protection différentielle",
                        "similarity_score": 0.85,
                        "weighted_score": 0.88
                    }
                ],
                "definitions": [
                    {
                        "content": "Disjoncteur différentiel: Dispositif de protection détectant les fuites de courant...",
                        "source": "Vocabulaire_technique.json",
                        "title": "Définition disjoncteur",
                        "similarity_score": 0.75,
                        "weighted_score": 0.78
                    }
                ]
            }
        }
        
        builder = ContextBuilder()
        context_result = builder.build_context(test_retrieval_results)
        
        print("✅ Construction contexte réussie")
        print(f"📊 Métadonnées: {context_result['metadata']}")
        print(f"📝 Aperçu contexte: {context_result['context_text'][:200]}...")
        print(f"🔧 Sections: {len(context_result['sections_details'])}")
        
        print("✅ Test passé avec succès")
        
    except Exception as e:
        print(f"❌ Test échoué: {e}")

In [ ]:
"""
Pipeline RAG complet pour l'analyse d'observations électriques
Intégration Retriever + ContextBuilder + LLM pour génération de réponses expertes
"""

import logging
import json
from typing import Dict, Any, Optional, List
from dataclasses import dataclass
from config import (
    SYSTEM_PROMPT,
    RAG_PROMPT_TEMPLATE,
    #OBSERVATION_PROMPT,
    LLM_MODEL,
    TOP_K
)

# Import conditionnel pour OpenAI
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI non disponible. Installation: pip install openai")

from retriever import get_retriever, retrieve_for_observation
from context_builder import get_context_builder, build_rag_context

# Configuration du logging
logger = logging.getLogger(__name__)

@dataclass
class PipelineResponse:
    """Réponse structurée du pipeline RAG"""
    observation_corrigee: str
    terminologie_enrichie: List[str]
    normes_applicables: List[str]
    analyse_technique: str
    recommandations: List[str]
    contexte_utilise: Dict[str, Any]
    metadata: Dict[str, Any]

class RAGPipeline:
    """
    Pipeline RAG complet pour l'analyse d'observations électriques
    """
    
    def __init__(self, retriever=None, context_builder=None, llm_model: str = None):
        """
        Initialise le pipeline RAG
        
        Args:
            retriever: Instance de Retriever
            context_builder: Instance de ContextBuilder
            llm_model: Modèle LLM à utiliser
        """
        self.retriever = retriever or get_retriever()
        self.context_builder = context_builder or get_context_builder()
        self.llm_model = llm_model or LLM_MODEL
        self.llm_client = None
        
        self._initialize_llm()
        logger.info(f"✅ Pipeline RAG initialisé avec modèle: {self.llm_model}")
    
    def _initialize_llm(self) -> None:
        """Initialise le client LLM"""
        if self.llm_model.startswith("gpt-"):
            if not OPENAI_AVAILABLE:
                raise ImportError("OpenAI non disponible. pip install openai")
            # Le client OpenAI sera initialisé avec la clé API de l'utilisateur
            self.llm_client = "openai"
        else:
            # Support pour d'autres modèles (LLaMA, Mistral, etc.)
            self.llm_client = "custom"
            logger.warning("⚠️  Modèle personnalisé - implémentez votre client LLM")
    
    def process_observation(self, observation: str, k: int = None) -> PipelineResponse:
        """
        Traite une observation électrique complète
        
        Args:
            observation: Observation technique brute
            k: Nombre de documents à récupérer
            
        Returns:
            Réponse structurée du pipeline
        """
        try:
            logger.info(f"🔧 Traitement observation: '{observation[:50]}...'")
            
            # Étape 1: Recherche des documents pertinents
            retrieval_results = self.retriever.retrieve_for_observation(observation, k)
            
            # Étape 2: Construction du contexte RAG
            context_result = self.context_builder.build_context(retrieval_results)
            
            # Étape 3: Génération de la réponse par LLM
            llm_response = self._generate_llm_response(observation, context_result)
            
            # Étape 4: Structuration de la réponse finale
            final_response = self._structure_final_response(
                observation, llm_response, context_result, retrieval_results
            )
            
            logger.info("✅ Traitement observation terminé avec succès")
            return final_response
            
        except Exception as e:
            logger.error(f"❌ Erreur traitement observation: {e}")
            return self._build_error_response(observation, str(e))
    
    def _generate_llm_response(self, observation: str, context_result: Dict[str, Any]) -> Dict[str, Any]:
        """
        Génère une réponse via le LLM
        """
        if not context_result["metadata"]["is_valid"]:
            return self._generate_fallback_response(observation)
        
        try:
            # Construction du prompt optimisé
            prompt = self._build_optimized_prompt(observation, context_result)
            
            # Appel au LLM
            if self.llm_client == "openai":
                llm_output = self._call_openai(prompt)
            else:
                llm_output = self._call_custom_llm(prompt)
            
            # Parsing de la réponse
            parsed_response = self._parse_llm_response(llm_output)
            
            return parsed_response
            
        except Exception as e:
            logger.error(f"❌ Erreur génération LLM: {e}")
            return self._generate_fallback_response(observation)
    
    def _build_optimized_prompt(self, observation: str, context_result: Dict[str, Any]) -> str:
        """
        Construit un prompt optimisé pour l'analyse électrique
        """
        context_text = context_result["context_text"]
        
        prompt = f"""
{SYSTEM_PROMPT}

{OBSERVATION_PROMPT.format(observation=observation, context=context_text)}

Votre analyse doit structurer la réponse en respectant STRICTEMENT le format JSON suivant :

{{
    "observation_corrigee": "Observation reformulée avec terminologie technique exacte",
    "terminologie_enrichie": ["liste", "des", "termes", "techniques", "pertinents"],
    "normes_applicables": ["NF C 15-100 § XXX", "NF C 15-100 § YYY"],
    "analyse_technique": "Analyse détaillée du problème et des risques",
    "recommandations": ["Action corrective 1", "Action corrective 2"]
}}

Exigences supplémentaires :
- Utilisez UNIQUEMENT le contexte fourni pour les références normatives
- Soyez technique et précis
- Proposez des actions concrètes et réalisables
- Mentionnez les risques de non-conformité

RÉPONSE (format JSON uniquement):
"""
        return prompt
    
    def _call_openai(self, prompt: str) -> str:
        """
        Appelle l'API OpenAI
        """
        # Note: L'utilisateur doit configurer sa clé API OpenAI
        # openai.api_key = "votre_cle_api"
        
        try:
            response = openai.ChatCompletion.create(
                model=self.llm_model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,  # Faible température pour des réponses stables
                max_tokens=1500,
                response_format={"type": "json_object"}  # Forcer le format JSON
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            logger.error(f"❌ Erreur appel OpenAI: {e}")
            raise
    
    def _call_custom_llm(self, prompt: str) -> str:
        """
        Appelle un modèle LLM personnalisé
        """
        # Implémentation à adapter selon le modèle utilisé
        # Exemple pour LLaMA, Mistral, etc.
        logger.warning("⚠️  Implémentez votre client LLM personnalisé")
        
        # Simulation de réponse pour les tests
        return json.dumps({
            "observation_corrigee": "Absence de disjoncteur différentiel 30mA de type A",
            "terminologie_enrichie": ["disjoncteur différentiel", "sensibilité 30mA", "type A", "protection personnes"],
            "normes_applicables": ["NF C 15-100 § 415.1", "NF C 15-100 § 531.2"],
            "analyse_technique": "L'absence de protection différentielle expose les personnes aux risques de électrocution par contact indirect. La norme exige un dispositif différentiel pour tous les circuits terminaux.",
            "recommandations": [
                "Installer un disjoncteur différentiel 30mA type A en tête des circuits concernés",
                "Vérifier la continuité des conducteurs de protection",
                "Contrôler la valeur de la prise de terre"
            ]
        })
    
    def _parse_llm_response(self, llm_output: str) -> Dict[str, Any]:
        """
        Parse la réponse du LLM en structure de données
        """
        try:
            # Nettoyer la réponse
            clean_output = llm_output.strip()
            if clean_output.startswith("```json"):
                clean_output = clean_output[7:]
            if clean_output.endswith("```"):
                clean_output = clean_output[:-3]
            
            # Parsing JSON
            parsed = json.loads(clean_output)
            
            # Validation des champs requis
            required_fields = [
                "observation_corrigee", "terminologie_enrichie", 
                "normes_applicables", "analyse_technique", "recommandations"
            ]
            
            for field in required_fields:
                if field not in parsed:
                    parsed[field] = "" if field != "recommandations" else []
            
            return parsed
            
        except json.JSONDecodeError as e:
            logger.error(f"❌ Erreur parsing JSON LLM: {e}")
            logger.debug(f"Contenu LLM: {llm_output}")
            return self._generate_fallback_parsing(llm_output)
    
    def _generate_fallback_response(self, observation: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours si le contexte est invalide
        """
        return {
            "observation_corrigee": observation,
            "terminologie_enrichie": ["observation à analyser"],
            "normes_applicables": ["NF C 15-100 (référence générale)"],
            "analyse_technique": "Impossible d'analyser cette observation avec les données disponibles. Vérifiez la formulation ou consultez un expert.",
            "recommandations": [
                "Reformuler l'observation avec plus de détails techniques",
                "Consulter la documentation normative NF C 15-100",
                "Contacter un organisme de certification"
            ]
        }
    
    def _generate_fallback_parsing(self, llm_output: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours en cas d'erreur de parsing
        """
        return {
            "observation_corrigee": "Analyse nécessitant validation manuelle",
            "terminologie_enrichie": ["validation expert requise"],
            "normes_applicables": ["NF C 15-100 § applicable"],
            "analyse_technique": f"Réponse LLM nécessitant interprétation: {llm_output[:200]}...",
            "recommandations": [
                "Faire vérifier l'analyse par un expert qualifié",
                "Consulter les documents normatifs directement"
            ]
        }
    
    def _structure_final_response(self, original_observation: str, llm_response: Dict[str, Any],
                                context_result: Dict[str, Any], retrieval_results: Dict[str, Any]) -> PipelineResponse:
        """
        Structure la réponse finale du pipeline
        """
        # Métadonnées de traitement
        metadata = {
            "contexte_valide": context_result["metadata"]["is_valid"],
            "documents_utilises": context_result["metadata"]["total_sections"],
            "types_documents": context_result["metadata"]["document_types"],
            "score_moyen": context_result["metadata"]["average_score"],
            "modele_llm": self.llm_model,
            "categorie_detectee": retrieval_results.get("categorie_detectee", "inconnue")
        }
        
        # Informations de contexte utilisées
        contexte_utilise = {
            "sections_count": len(context_result["sections_details"]),
            "sources": list(set(section["source"] for section in context_result["sections_details"])),
            "contexte_length": context_result["metadata"]["context_length"]
        }
        
        return PipelineResponse(
            observation_corrigee=llm_response["observation_corrigee"],
            terminologie_enrichie=llm_response["terminologie_enrichie"],
            normes_applicables=llm_response["normes_applicables"],
            analyse_technique=llm_response["analyse_technique"],
            recommandations=llm_response["recommandations"],
            contexte_utilise=contexte_utilise,
            metadata=metadata
        )
    
    def _build_error_response(self, observation: str, error_msg: str) -> PipelineResponse:
        """
        Construit une réponse d'erreur
        """
        return PipelineResponse(
            observation_corrigee=observation,
            terminologie_enrichie=["erreur traitement"],
            normes_applicables=[],
            analyse_technique=f"Erreur lors du traitement: {error_msg}",
            recommandations=["Vérifier la configuration du pipeline", "Contacter le support technique"],
            contexte_utilise={},
            metadata={"erreur": error_msg, "contexte_valide": False}
        )
    
    def generate_report(self, response: PipelineResponse, format: str = "text") -> str:
        """
        Génère un rapport formaté à partir de la réponse
        
        Args:
            response: Réponse du pipeline
            format: Format de sortie ('text', 'markdown', 'json')
            
        Returns:
            Rapport formaté
        """
        if format == "json":
            return json.dumps({
                "observation_corrigee": response.observation_corrigee,
                "terminologie_enrichie": response.terminologie_enrichie,
                "normes_applicables": response.normes_applicables,
                "analyse_technique": response.analyse_technique,
                "recommandations": response.recommandations,
                "metadata": response.metadata
            }, indent=2, ensure_ascii=False)
        
        elif format == "markdown":
            return self._generate_markdown_report(response)
        
        else:  # format text
            return self._generate_text_report(response)
    
    def _generate_markdown_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format Markdown"""
        report = [
            "# RAPPORT D'ANALYSE ÉLECTRIQUE",
            "",
            f"## Observation corrigée",
            f"{response.observation_corrigee}",
            "",
            f"## Terminologie technique",
            "\n".join(f"- {term}" for term in response.terminologie_enrichie),
            "",
            f"## Normes applicables",
            "\n".join(f"- {norme}" for norme in response.normes_applicables),
            "",
            f"## Analyse technique",
            f"{response.analyse_technique}",
            "",
            f"## Recommandations",
            "\n".join(f"{i+1}. {reco}" for i, reco in enumerate(response.recommandations)),
            "",
            f"## Métadonnées",
            f"- Contexte valide: {response.metadata.get('contexte_valide', 'N/A')}",
            f"- Documents utilisés: {response.metadata.get('documents_utilises', 'N/A')}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ]
        
        return "\n".join(report)
    
    def _generate_text_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format texte simple"""
        report = [
            "RAPPORT D'ANALYSE ÉLECTRIQUE",
            "=" * 50,
            "",
            f"OBSERVATION CORRIGÉE:",
            f"{response.observation_corrigee}",
            "",
            f"TERMINOLOGIE TECHNIQUE:",
            f"{', '.join(response.terminologie_enrichie)}",
            "",
            f"NORMES APPLICABLES:",
            f"{', '.join(response.normes_applicables)}",
            "",
            f"ANALYSE TECHNIQUE:",
            f"{response.analyse_technique}",
            "",
            f"RECOMMANDATIONS:",
        ]
        
        for i, reco in enumerate(response.recommandations, 1):
            report.append(f"{i}. {reco}")
        
        report.extend([
            "",
            f"INFORMATIONS:",
            f"- Sources utilisées: {len(response.contexte_utilise.get('sources', []))}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ])
        
        return "\n".join(report)


# Instance singleton pour une utilisation facile
_pipeline_instance = None

def get_pipeline() -> RAGPipeline:
    """
    Retourne une instance singleton du pipeline RAG
    
    Returns:
        Instance de RAGPipeline
    """
    global _pipeline_instance
    
    if _pipeline_instance is None:
        _pipeline_instance = RAGPipeline()
    
    return _pipeline_instance

def process_electrical_observation(observation: str, k: int = None) -> PipelineResponse:
    """
    Fonction utilitaire pour traitement rapide d'observation
    
    Args:
        observation: Observation technique
        k: Nombre de documents
        
    Returns:
        Réponse structurée
    """
    pipeline = get_pipeline()
    return pipeline.process_observation(observation, k)

# Test du module
if __name__ == "__main__":
    print("🧪 Test du Pipeline RAG...")
    
    try:
        # Configuration pour test (sans appel réel à OpenAI)
        import os
        os.environ["OPENAI_API_KEY"] = "test-key"  # Clé factice pour test
        
        pipeline = RAGPipeline(llm_model="gpt-3.5-turbo")
        
        # Test avec une observation électrique
        test_observation = "Disjoncteur différentiel 30mA manquant dans le tableau cuisine"
        response = pipeline.process_observation(test_observation)
        
        print("✅ Pipeline RAG testé avec succès")
        print(f"📊 Observation corrigée: {response.observation_corrigee}")
        print(f"🔧 Normes applicables: {response.normes_applicables}")
        print(f"📝 Recommandations: {len(response.recommandations)}")
        
        # Génération de rapport
        report = pipeline.generate_report(response, "text")
        print(f"\n📄 Rapport généré ({len(report)} caractères)")
        
        print("✅ Test passé avec succès")
        
    except Exception as e:
        print(f"❌ Test échoué: {e}")

In [ ]:
from rag_pipeline import get_pipeline, process_electrical_observation

# Méthode 1: Instance complète
pipeline = get_pipeline()
observation = "Prise sans terre dans salle de bain"
response = pipeline.process_observation(observation)

print(f"✅ Observation corrigée: {response.observation_corrigee}")
print(f"📚 Normes: {response.normes_applicables}")
print(f"🔧 Recommandations: {response.recommandations}")

# Génération de rapport
report = pipeline.generate_report(response, "markdown")
print(report)

# Méthode 2: Fonction utilitaire
response = process_electrical_observation("Câble dénudé dans gaine")

In [ ]:
"""
Pipeline RAG complet pour l'analyse d'observations électriques
Intégration Retriever + ContextBuilder + LLM pour génération de réponses expertes
"""

import logging
import json
from typing import Dict, Any, Optional, List
from dataclasses import dataclass
from config import (
    SYSTEM_PROMPT,
    RAG_PROMPT_TEMPLATE,
    OBSERVATION_PROMPT,
    LLM_MODEL,
    TOP_K
)

# Import conditionnel pour OpenAI
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI non disponible. Installation: pip install openai")

from retriever import get_retriever, retrieve_for_observation
from context_builder import get_context_builder, build_rag_context

# Configuration du logging
logger = logging.getLogger(__name__)

@dataclass
class PipelineResponse:
    """Réponse structurée du pipeline RAG"""
    observation_corrigee: str
    terminologie_enrichie: List[str]
    normes_applicables: List[str]
    analyse_technique: str
    recommandations: List[str]
    contexte_utilise: Dict[str, Any]
    metadata: Dict[str, Any]

class RAGPipeline:
    """
    Pipeline RAG complet pour l'analyse d'observations électriques
    """
    
    def __init__(self, retriever=None, context_builder=None, llm_model: str = None):
        """
        Initialise le pipeline RAG
        
        Args:
            retriever: Instance de Retriever
            context_builder: Instance de ContextBuilder
            llm_model: Modèle LLM à utiliser
        """
        self.retriever = retriever or get_retriever()
        self.context_builder = context_builder or get_context_builder()
        self.llm_model = llm_model or LLM_MODEL
        self.llm_client = None
        
        self._initialize_llm()
        logger.info(f"✅ Pipeline RAG initialisé avec modèle: {self.llm_model}")
    
    def _initialize_llm(self) -> None:
        """Initialise le client LLM"""
        if self.llm_model.startswith("gpt-"):
            if not OPENAI_AVAILABLE:
                raise ImportError("OpenAI non disponible. pip install openai")
            # Le client OpenAI sera initialisé avec la clé API de l'utilisateur
            self.llm_client = "openai"
        else:
            # Support pour d'autres modèles (LLaMA, Mistral, etc.)
            self.llm_client = "custom"
            logger.warning("⚠️  Modèle personnalisé - implémentez votre client LLM")
    
    def process_observation(self, observation: str, k: int = None) -> PipelineResponse:
        """
        Traite une observation électrique complète
        
        Args:
            observation: Observation technique brute
            k: Nombre de documents à récupérer
            
        Returns:
            Réponse structurée du pipeline
        """
        try:
            logger.info(f"🔧 Traitement observation: '{observation[:50]}...'")
            
            # Étape 1: Recherche des documents pertinents
            retrieval_results = self.retriever.retrieve_for_observation(observation, k)
            
            # Étape 2: Construction du contexte RAG
            context_result = self.context_builder.build_context(retrieval_results)
            
            # Étape 3: Génération de la réponse par LLM
            llm_response = self._generate_llm_response(observation, context_result)
            
            # Étape 4: Structuration de la réponse finale
            final_response = self._structure_final_response(
                observation, llm_response, context_result, retrieval_results
            )
            
            logger.info("✅ Traitement observation terminé avec succès")
            return final_response
            
        except Exception as e:
            logger.error(f"❌ Erreur traitement observation: {e}")
            return self._build_error_response(observation, str(e))
    
    def _generate_llm_response(self, observation: str, context_result: Dict[str, Any]) -> Dict[str, Any]:
        """
        Génère une réponse via le LLM
        """
        if not context_result["metadata"]["is_valid"]:
            return self._generate_fallback_response(observation)
        
        try:
            # Construction du prompt optimisé
            prompt = self._build_optimized_prompt(observation, context_result)
            
            # Appel au LLM
            if self.llm_client == "openai":
                llm_output = self._call_openai(prompt)
            else:
                llm_output = self._call_custom_llm(prompt)
            
            # Parsing de la réponse
            parsed_response = self._parse_llm_response(llm_output)
            
            return parsed_response
            
        except Exception as e:
            logger.error(f"❌ Erreur génération LLM: {e}")
            return self._generate_fallback_response(observation)
    
    def _build_optimized_prompt(self, observation: str, context_result: Dict[str, Any]) -> str:
        """
        Construit un prompt optimisé pour l'analyse électrique
        """
        context_text = context_result["context_text"]
        
        prompt = f"""
{SYSTEM_PROMPT}

{OBSERVATION_PROMPT.format(observation=observation, context=context_text)}

Votre analyse doit structurer la réponse en respectant STRICTEMENT le format JSON suivant :

{{
    "observation_corrigee": "Observation reformulée avec terminologie technique exacte",
    "terminologie_enrichie": ["liste", "des", "termes", "techniques", "pertinents"],
    "normes_applicables": ["NF C 15-100 § XXX", "NF C 15-100 § YYY"],
    "analyse_technique": "Analyse détaillée du problème et des risques",
    "recommandations": ["Action corrective 1", "Action corrective 2"]
}}

Exigences supplémentaires :
- Utilisez UNIQUEMENT le contexte fourni pour les références normatives
- Soyez technique et précis
- Proposez des actions concrètes et réalisables
- Mentionnez les risques de non-conformité

RÉPONSE (format JSON uniquement):
"""
        return prompt
    
    def _call_openai(self, prompt: str) -> str:
        """
        Appelle l'API OpenAI
        """
        # Note: L'utilisateur doit configurer sa clé API OpenAI
        # openai.api_key = "votre_cle_api"
        
        try:
            response = openai.ChatCompletion.create(
                model=self.llm_model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,  # Faible température pour des réponses stables
                max_tokens=1500,
                response_format={"type": "json_object"}  # Forcer le format JSON
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            logger.error(f"❌ Erreur appel OpenAI: {e}")
            raise
    
    def _call_custom_llm(self, prompt: str) -> str:
        """
        Appelle un modèle LLM personnalisé
        """
        # Implémentation à adapter selon le modèle utilisé
        # Exemple pour LLaMA, Mistral, etc.
        logger.warning("⚠️  Implémentez votre client LLM personnalisé")
        
        # Simulation de réponse pour les tests
        return json.dumps({
            "observation_corrigee": "Absence de disjoncteur différentiel 30mA de type A",
            "terminologie_enrichie": ["disjoncteur différentiel", "sensibilité 30mA", "type A", "protection personnes"],
            "normes_applicables": ["NF C 15-100 § 415.1", "NF C 15-100 § 531.2"],
            "analyse_technique": "L'absence de protection différentielle expose les personnes aux risques de électrocution par contact indirect. La norme exige un dispositif différentiel pour tous les circuits terminaux.",
            "recommandations": [
                "Installer un disjoncteur différentiel 30mA type A en tête des circuits concernés",
                "Vérifier la continuité des conducteurs de protection",
                "Contrôler la valeur de la prise de terre"
            ]
        })
    
    def _parse_llm_response(self, llm_output: str) -> Dict[str, Any]:
        """
        Parse la réponse du LLM en structure de données
        """
        try:
            # Nettoyer la réponse
            clean_output = llm_output.strip()
            if clean_output.startswith("```json"):
                clean_output = clean_output[7:]
            if clean_output.endswith("```"):
                clean_output = clean_output[:-3]
            
            # Parsing JSON
            parsed = json.loads(clean_output)
            
            # Validation des champs requis
            required_fields = [
                "observation_corrigee", "terminologie_enrichie", 
                "normes_applicables", "analyse_technique", "recommandations"
            ]
            
            for field in required_fields:
                if field not in parsed:
                    parsed[field] = "" if field != "recommandations" else []
            
            return parsed
            
        except json.JSONDecodeError as e:
            logger.error(f"❌ Erreur parsing JSON LLM: {e}")
            logger.debug(f"Contenu LLM: {llm_output}")
            return self._generate_fallback_parsing(llm_output)
    
    def _generate_fallback_response(self, observation: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours si le contexte est invalide
        """
        return {
            "observation_corrigee": observation,
            "terminologie_enrichie": ["observation à analyser"],
            "normes_applicables": ["NF C 15-100 (référence générale)"],
            "analyse_technique": "Impossible d'analyser cette observation avec les données disponibles. Vérifiez la formulation ou consultez un expert.",
            "recommandations": [
                "Reformuler l'observation avec plus de détails techniques",
                "Consulter la documentation normative NF C 15-100",
                "Contacter un organisme de certification"
            ]
        }
    
    def _generate_fallback_parsing(self, llm_output: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours en cas d'erreur de parsing
        """
        return {
            "observation_corrigee": "Analyse nécessitant validation manuelle",
            "terminologie_enrichie": ["validation expert requise"],
            "normes_applicables": ["NF C 15-100 § applicable"],
            "analyse_technique": f"Réponse LLM nécessitant interprétation: {llm_output[:200]}...",
            "recommandations": [
                "Faire vérifier l'analyse par un expert qualifié",
                "Consulter les documents normatifs directement"
            ]
        }
    
    def _structure_final_response(self, original_observation: str, llm_response: Dict[str, Any],
                                context_result: Dict[str, Any], retrieval_results: Dict[str, Any]) -> PipelineResponse:
        """
        Structure la réponse finale du pipeline
        """
        # Métadonnées de traitement
        metadata = {
            "contexte_valide": context_result["metadata"]["is_valid"],
            "documents_utilises": context_result["metadata"]["total_sections"],
            "types_documents": context_result["metadata"]["document_types"],
            "score_moyen": context_result["metadata"]["average_score"],
            "modele_llm": self.llm_model,
            "categorie_detectee": retrieval_results.get("categorie_detectee", "inconnue")
        }
        
        # Informations de contexte utilisées
        contexte_utilise = {
            "sections_count": len(context_result["sections_details"]),
            "sources": list(set(section["source"] for section in context_result["sections_details"])),
            "contexte_length": context_result["metadata"]["context_length"]
        }
        
        return PipelineResponse(
            observation_corrigee=llm_response["observation_corrigee"],
            terminologie_enrichie=llm_response["terminologie_enrichie"],
            normes_applicables=llm_response["normes_applicables"],
            analyse_technique=llm_response["analyse_technique"],
            recommandations=llm_response["recommandations"],
            contexte_utilise=contexte_utilise,
            metadata=metadata
        )
    
    def _build_error_response(self, observation: str, error_msg: str) -> PipelineResponse:
        """
        Construit une réponse d'erreur
        """
        return PipelineResponse(
            observation_corrigee=observation,
            terminologie_enrichie=["erreur traitement"],
            normes_applicables=[],
            analyse_technique=f"Erreur lors du traitement: {error_msg}",
            recommandations=["Vérifier la configuration du pipeline", "Contacter le support technique"],
            contexte_utilise={},
            metadata={"erreur": error_msg, "contexte_valide": False}
        )
    
    def generate_report(self, response: PipelineResponse, format: str = "text") -> str:
        """
        Génère un rapport formaté à partir de la réponse
        
        Args:
            response: Réponse du pipeline
            format: Format de sortie ('text', 'markdown', 'json')
            
        Returns:
            Rapport formaté
        """
        if format == "json":
            return json.dumps({
                "observation_corrigee": response.observation_corrigee,
                "terminologie_enrichie": response.terminologie_enrichie,
                "normes_applicables": response.normes_applicables,
                "analyse_technique": response.analyse_technique,
                "recommandations": response.recommandations,
                "metadata": response.metadata
            }, indent=2, ensure_ascii=False)
        
        elif format == "markdown":
            return self._generate_markdown_report(response)
        
        else:  # format text
            return self._generate_text_report(response)
    
    def _generate_markdown_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format Markdown"""
        report = [
            "# RAPPORT D'ANALYSE ÉLECTRIQUE",
            "",
            f"## Observation corrigée",
            f"{response.observation_corrigee}",
            "",
            f"## Terminologie technique",
            "\n".join(f"- {term}" for term in response.terminologie_enrichie),
            "",
            f"## Normes applicables",
            "\n".join(f"- {norme}" for norme in response.normes_applicables),
            "",
            f"## Analyse technique",
            f"{response.analyse_technique}",
            "",
            f"## Recommandations",
            "\n".join(f"{i+1}. {reco}" for i, reco in enumerate(response.recommandations)),
            "",
            f"## Métadonnées",
            f"- Contexte valide: {response.metadata.get('contexte_valide', 'N/A')}",
            f"- Documents utilisés: {response.metadata.get('documents_utilises', 'N/A')}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ]
        
        return "\n".join(report)
    
    def _generate_text_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format texte simple"""
        report = [
            "RAPPORT D'ANALYSE ÉLECTRIQUE",
            "=" * 50,
            "",
            f"OBSERVATION CORRIGÉE:",
            f"{response.observation_corrigee}",
            "",
            f"TERMINOLOGIE TECHNIQUE:",
            f"{', '.join(response.terminologie_enrichie)}",
            "",
            f"NORMES APPLICABLES:",
            f"{', '.join(response.normes_applicables)}",
            "",
            f"ANALYSE TECHNIQUE:",
            f"{response.analyse_technique}",
            "",
            f"RECOMMANDATIONS:",
        ]
        
        for i, reco in enumerate(response.recommandations, 1):
            report.append(f"{i}. {reco}")
        
        report.extend([
            "",
            f"INFORMATIONS:",
            f"- Sources utilisées: {len(response.contexte_utilise.get('sources', []))}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ])
        
        return "\n".join(report)


# Instance singleton pour une utilisation facile
_pipeline_instance = None

def get_pipeline() -> RAGPipeline:
    """
    Retourne une instance singleton du pipeline RAG
    
    Returns:
        Instance de RAGPipeline
    """
    global _pipeline_instance
    
    if _pipeline_instance is None:
        _pipeline_instance = RAGPipeline()
    
    return _pipeline_instance

def process_electrical_observation(observation: str, k: int = None) -> PipelineResponse:
    """
    Fonction utilitaire pour traitement rapide d'observation
    
    Args:
        observation: Observation technique
        k: Nombre de documents
        
    Returns:
        Réponse structurée
    """
    pipeline = get_pipeline()
    return pipeline.process_observation(observation, k)

# Test du module
if __name__ == "__main__":
    print("🧪 Test du Pipeline RAG...")
    
    try:
        # Configuration pour test (sans appel réel à OpenAI)
        import os
        os.environ["OPENAI_API_KEY"] = "test-key"  # Clé factice pour test
        
        pipeline = RAGPipeline(llm_model="gpt-3.5-turbo")
        
        # Test avec une observation électrique
        test_observation = "Disjoncteur différentiel 30mA manquant dans le tableau cuisine"
        response = pipeline.process_observation(test_observation)
        
        print("✅ Pipeline RAG testé avec succès")
        print(f"📊 Observation corrigée: {response.observation_corrigee}")
        print(f"🔧 Normes applicables: {response.normes_applicables}")
        print(f"📝 Recommandations: {len(response.recommandations)}")
        
        # Génération de rapport
        report = pipeline.generate_report(response, "text")
        print(f"\n📄 Rapport généré ({len(report)} caractères)")
        
        print("✅ Test passé avec succès")
        
    except Exception as e:
        print(f"❌ Test échoué: {e}")

In [ ]:
"""
Pipeline RAG complet pour l'analyse d'observations électriques
Intégration Retriever + ContextBuilder + LLM pour génération de réponses expertes
"""

import logging
import json
from typing import Dict, Any, Optional, List
from dataclasses import dataclass
from config import (
    SYSTEM_PROMPT,
    RAG_PROMPT_TEMPLATE,
    OBSERVATION_PROMPT,
    LLM_MODEL,
    TOP_K
)

# Import conditionnel pour OpenAI
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI non disponible. Installation: pip install openai")

# Import des modules corrigés
try:
    from retriever import get_retriever, retrieve_for_observation
    from context_builder import get_context_builder, build_rag_context
except ImportError as e:
    print(f"❌ Erreur importation modules: {e}")
    # Définir des fonctions factices pour permettre le test
    def get_retriever():
        raise ImportError("Retriever non disponible")
    def retrieve_for_observation(*args, **kwargs):
        return {}
    def get_context_builder():
        raise ImportError("ContextBuilder non disponible") 
    def build_rag_context(*args, **kwargs):
        return {}

# Configuration du logging
logger = logging.getLogger(__name__)

@dataclass
class PipelineResponse:
    """Réponse structurée du pipeline RAG"""
    observation_corrigee: str
    terminologie_enrichie: List[str]
    normes_applicables: List[str]
    analyse_technique: str
    recommandations: List[str]
    contexte_utilise: Dict[str, Any]
    metadata: Dict[str, Any]

class RAGPipeline:
    """
    Pipeline RAG complet pour l'analyse d'observations électriques
    """
    
    def __init__(self, retriever=None, context_builder=None, llm_model: str = None):
        """
        Initialise le pipeline RAG
        
        Args:
            retriever: Instance de Retriever
            context_builder: Instance de ContextBuilder
            llm_model: Modèle LLM à utiliser
        """
        self.retriever = retriever or get_retriever()
        self.context_builder = context_builder or get_context_builder()
        self.llm_model = llm_model or LLM_MODEL
        self.llm_client = None
        
        self._initialize_llm()
        logger.info(f"✅ Pipeline RAG initialisé avec modèle: {self.llm_model}")
    
    def _initialize_llm(self) -> None:
        """Initialise le client LLM"""
        if self.llm_model.startswith("gpt-"):
            if not OPENAI_AVAILABLE:
                raise ImportError("OpenAI non disponible. pip install openai")
            # Le client OpenAI sera initialisé avec la clé API de l'utilisateur
            self.llm_client = "openai"
        else:
            # Support pour d'autres modèles (LLaMA, Mistral, etc.)
            self.llm_client = "custom"
            logger.warning("⚠️  Modèle personnalisé - implémentez votre client LLM")
    
    def process_observation(self, observation: str, k: int = None) -> PipelineResponse:
        """
        Traite une observation électrique complète
        
        Args:
            observation: Observation technique brute
            k: Nombre de documents à récupérer
            
        Returns:
            Réponse structurée du pipeline
        """
        try:
            logger.info(f"🔧 Traitement observation: '{observation[:50]}...'")
            
            # Étape 1: Recherche des documents pertinents
            retrieval_results = self.retriever.retrieve_for_observation(observation, k)
            
            # Étape 2: Construction du contexte RAG
            context_result = self.context_builder.build_context(retrieval_results)
            
            # Étape 3: Génération de la réponse par LLM
            llm_response = self._generate_llm_response(observation, context_result)
            
            # Étape 4: Structuration de la réponse finale
            final_response = self._structure_final_response(
                observation, llm_response, context_result, retrieval_results
            )
            
            logger.info("✅ Traitement observation terminé avec succès")
            return final_response
            
        except Exception as e:
            logger.error(f"❌ Erreur traitement observation: {e}")
            return self._build_error_response(observation, str(e))
    
    def _generate_llm_response(self, observation: str, context_result: Dict[str, Any]) -> Dict[str, Any]:
        """
        Génère une réponse via le LLM
        """
        if not context_result["metadata"]["is_valid"]:
            return self._generate_fallback_response(observation)
        
        try:
            # Construction du prompt optimisé
            prompt = self._build_optimized_prompt(observation, context_result)
            
            # Appel au LLM
            if self.llm_client == "openai":
                llm_output = self._call_openai(prompt)
            else:
                llm_output = self._call_custom_llm(prompt)
            
            # Parsing de la réponse
            parsed_response = self._parse_llm_response(llm_output)
            
            return parsed_response
            
        except Exception as e:
            logger.error(f"❌ Erreur génération LLM: {e}")
            return self._generate_fallback_response(observation)
    
    def _build_optimized_prompt(self, observation: str, context_result: Dict[str, Any]) -> str:
        """
        Construit un prompt optimisé pour l'analyse électrique
        """
        context_text = context_result["context_text"]
        
        prompt = f"""
{SYSTEM_PROMPT}

{OBSERVATION_PROMPT.format(observation=observation, context=context_text)}

Votre analyse doit structurer la réponse en respectant STRICTEMENT le format JSON suivant :

{{
    "observation_corrigee": "Observation reformulée avec terminologie technique exacte",
    "terminologie_enrichie": ["liste", "des", "termes", "techniques", "pertinents"],
    "normes_applicables": ["NF C 15-100 § XXX", "NF C 15-100 § YYY"],
    "analyse_technique": "Analyse détaillée du problème et des risques",
    "recommandations": ["Action corrective 1", "Action corrective 2"]
}}

Exigences supplémentaires :
- Utilisez UNIQUEMENT le contexte fourni pour les références normatives
- Soyez technique et précis
- Proposez des actions concrètes et réalisables
- Mentionnez les risques de non-conformité

RÉPONSE (format JSON uniquement):
"""
        return prompt
    
    def _call_openai(self, prompt: str) -> str:
        """
        Appelle l'API OpenAI
        """
        # Note: L'utilisateur doit configurer sa clé API OpenAI
        # openai.api_key = "votre_cle_api"
        
        try:
            # Pour les nouvelles versions de l'API OpenAI
            if hasattr(openai, 'ChatCompletion'):
                response = openai.ChatCompletion.create(
                    model=self.llm_model,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1,
                    max_tokens=1500
                )
                return response.choices[0].message.content
            else:
                # Pour les versions plus récentes de l'API
                client = openai.OpenAI()
                response = client.chat.completions.create(
                    model=self.llm_model,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1,
                    max_tokens=1500
                )
                return response.choices[0].message.content
                
        except Exception as e:
            logger.error(f"❌ Erreur appel OpenAI: {e}")
            # Retourner une réponse simulée pour les tests
            return self._get_simulated_response()
    
    def _get_simulated_response(self) -> str:
        """Retourne une réponse simulée pour les tests"""
        return json.dumps({
            "observation_corrigee": "Absence de disjoncteur différentiel 30mA de type A dans le tableau électrique de la cuisine",
            "terminologie_enrichie": ["disjoncteur différentiel", "sensibilité 30mA", "type A", "protection personnes", "tableau électrique"],
            "normes_applicables": ["NF C 15-100 § 415.1", "NF C 15-100 § 531.2"],
            "analyse_technique": "L'absence de protection différentielle expose les personnes aux risques d'électrocution par contact indirect. La norme NF C 15-100 exige la présence d'un dispositif différentiel 30mA pour tous les circuits terminaux dans les locaux d'habitation.",
            "recommandations": [
                "Installer un disjoncteur différentiel 30mA type A en tête des circuits de la cuisine",
                "Vérifier la continuité des conducteurs de protection",
                "Contrôler la valeur de la prise de terre",
                "Réaliser une mesure de l'efficacité du différentiel"
            ]
        })
    
    def _call_custom_llm(self, prompt: str) -> str:
        """
        Appelle un modèle LLM personnalisé
        """
        # Implémentation à adapter selon le modèle utilisé
        # Exemple pour LLaMA, Mistral, etc.
        logger.warning("⚠️  Implémentez votre client LLM personnalisé")
        return self._get_simulated_response()
    
    def _parse_llm_response(self, llm_output: str) -> Dict[str, Any]:
        """
        Parse la réponse du LLM en structure de données
        """
        try:
            # Nettoyer la réponse
            clean_output = llm_output.strip()
            if clean_output.startswith("```json"):
                clean_output = clean_output[7:]
            if clean_output.endswith("```"):
                clean_output = clean_output[:-3]
            
            # Parsing JSON
            parsed = json.loads(clean_output)
            
            # Validation des champs requis
            required_fields = [
                "observation_corrigee", "terminologie_enrichie", 
                "normes_applicables", "analyse_technique", "recommandations"
            ]
            
            for field in required_fields:
                if field not in parsed:
                    if field == "recommandations":
                        parsed[field] = []
                    else:
                        parsed[field] = ""
            
            return parsed
            
        except json.JSONDecodeError as e:
            logger.error(f"❌ Erreur parsing JSON LLM: {e}")
            logger.debug(f"Contenu LLM: {llm_output}")
            return self._generate_fallback_parsing(llm_output)
    
    def _generate_fallback_response(self, observation: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours si le contexte est invalide
        """
        return {
            "observation_corrigee": observation,
            "terminologie_enrichie": ["observation à analyser"],
            "normes_applicables": ["NF C 15-100 (référence générale)"],
            "analyse_technique": "Impossible d'analyser cette observation avec les données disponibles. Vérifiez la formulation ou consultez un expert.",
            "recommandations": [
                "Reformuler l'observation avec plus de détails techniques",
                "Consulter la documentation normative NF C 15-100",
                "Contacter un organisme de certification"
            ]
        }
    
    def _generate_fallback_parsing(self, llm_output: str) -> Dict[str, Any]:
        """
        Génère une réponse de secours en cas d'erreur de parsing
        """
        return {
            "observation_corrigee": "Analyse nécessitant validation manuelle",
            "terminologie_enrichie": ["validation expert requise"],
            "normes_applicables": ["NF C 15-100 § applicable"],
            "analyse_technique": f"Réponse LLM nécessitant interprétation: {llm_output[:200]}...",
            "recommandations": [
                "Faire vérifier l'analyse par un expert qualifié",
                "Consulter les documents normatifs directement"
            ]
        }
    
    def _structure_final_response(self, original_observation: str, llm_response: Dict[str, Any],
                                context_result: Dict[str, Any], retrieval_results: Dict[str, Any]) -> PipelineResponse:
        """
        Structure la réponse finale du pipeline
        """
        # Métadonnées de traitement
        metadata = {
            "contexte_valide": context_result["metadata"]["is_valid"],
            "documents_utilises": context_result["metadata"]["total_sections"],
            "types_documents": context_result["metadata"]["document_types"],
            "score_moyen": context_result["metadata"]["average_score"],
            "modele_llm": self.llm_model,
            "categorie_detectee": retrieval_results.get("categorie_detectee", "inconnue")
        }
        
        # Informations de contexte utilisées
        contexte_utilise = {
            "sections_count": len(context_result["sections_details"]),
            "sources": list(set(section["source"] for section in context_result["sections_details"])),
            "contexte_length": context_result["metadata"]["context_length"]
        }
        
        return PipelineResponse(
            observation_corrigee=llm_response["observation_corrigee"],
            terminologie_enrichie=llm_response["terminologie_enrichie"],
            normes_applicables=llm_response["normes_applicables"],
            analyse_technique=llm_response["analyse_technique"],
            recommandations=llm_response["recommandations"],
            contexte_utilise=contexte_utilise,
            metadata=metadata
        )
    
    def _build_error_response(self, observation: str, error_msg: str) -> PipelineResponse:
        """
        Construit une réponse d'erreur
        """
        return PipelineResponse(
            observation_corrigee=observation,
            terminologie_enrichie=["erreur traitement"],
            normes_applicables=[],
            analyse_technique=f"Erreur lors du traitement: {error_msg}",
            recommandations=["Vérifier la configuration du pipeline", "Contacter le support technique"],
            contexte_utilise={},
            metadata={"erreur": error_msg, "contexte_valide": False}
        )
    
    def generate_report(self, response: PipelineResponse, format: str = "text") -> str:
        """
        Génère un rapport formaté à partir de la réponse
        
        Args:
            response: Réponse du pipeline
            format: Format de sortie ('text', 'markdown', 'json')
            
        Returns:
            Rapport formaté
        """
        if format == "json":
            return json.dumps({
                "observation_corrigee": response.observation_corrigee,
                "terminologie_enrichie": response.terminologie_enrichie,
                "normes_applicables": response.normes_applicables,
                "analyse_technique": response.analyse_technique,
                "recommandations": response.recommandations,
                "metadata": response.metadata
            }, indent=2, ensure_ascii=False)
        
        elif format == "markdown":
            return self._generate_markdown_report(response)
        
        else:  # format text
            return self._generate_text_report(response)
    
    def _generate_markdown_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format Markdown"""
        report = [
            "# RAPPORT D'ANALYSE ÉLECTRIQUE",
            "",
            f"## Observation corrigée",
            f"{response.observation_corrigee}",
            "",
            f"## Terminologie technique",
            "\n".join(f"- {term}" for term in response.terminologie_enrichie),
            "",
            f"## Normes applicables",
            "\n".join(f"- {norme}" for norme in response.normes_applicables),
            "",
            f"## Analyse technique",
            f"{response.analyse_technique}",
            "",
            f"## Recommandations",
            "\n".join(f"{i+1}. {reco}" for i, reco in enumerate(response.recommandations)),
            "",
            f"## Métadonnées",
            f"- Contexte valide: {response.metadata.get('contexte_valide', 'N/A')}",
            f"- Documents utilisés: {response.metadata.get('documents_utilises', 'N/A')}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ]
        
        return "\n".join(report)
    
    def _generate_text_report(self, response: PipelineResponse) -> str:
        """Génère un rapport au format texte simple"""
        report = [
            "RAPPORT D'ANALYSE ÉLECTRIQUE",
            "=" * 50,
            "",
            f"OBSERVATION CORRIGÉE:",
            f"{response.observation_corrigee}",
            "",
            f"TERMINOLOGIE TECHNIQUE:",
            f"{', '.join(response.terminologie_enrichie)}",
            "",
            f"NORMES APPLICABLES:",
            f"{', '.join(response.normes_applicables)}",
            "",
            f"ANALYSE TECHNIQUE:",
            f"{response.analyse_technique}",
            "",
            f"RECOMMANDATIONS:",
        ]
        
        for i, reco in enumerate(response.recommandations, 1):
            report.append(f"{i}. {reco}")
        
        report.extend([
            "",
            f"INFORMATIONS:",
            f"- Sources utilisées: {len(response.contexte_utilise.get('sources', []))}",
            f"- Catégorie: {response.metadata.get('categorie_detectee', 'N/A')}",
        ])
        
        return "\n".join(report)


# Instance singleton pour une utilisation facile
_pipeline_instance = None

def get_pipeline() -> RAGPipeline:
    """
    Retourne une instance singleton du pipeline RAG
    
    Returns:
        Instance de RAGPipeline
    """
    global _pipeline_instance
    
    if _pipeline_instance is None:
        _pipeline_instance = RAGPipeline()
    
    return _pipeline_instance

def process_electrical_observation(observation: str, k: int = None) -> PipelineResponse:
    """
    Fonction utilitaire pour traitement rapide d'observation
    
    Args:
        observation: Observation technique
        k: Nombre de documents
        
    Returns:
        Réponse structurée
    """
    pipeline = get_pipeline()
    return pipeline.process_observation(observation, k)

# Test du module
if __name__ == "__main__":
    print("🧪 Test du Pipeline RAG...")
    
    try:
        # Test avec une observation électrique (sans OpenAI)
        pipeline = RAGPipeline(llm_model="custom")  # Utiliser le mode custom pour éviter OpenAI
        
        test_observation = "Disjoncteur différentiel 30mA manquant dans le tableau cuisine"
        response = pipeline.process_observation(test_observation)
        
        print("✅ Pipeline RAG testé avec succès")
        print(f"📊 Observation corrigée: {response.observation_corrigee}")
        print(f"🔧 Normes applicables: {response.normes_applicables}")
        print(f"📝 Recommandations: {len(response.recommandations)}")
        
        # Génération de rapport
        report = pipeline.generate_report(response, "text")
        print(f"\n📄 Rapport généré:")
        print(report)
        
        print("✅ Test passé avec succès")
        
    except Exception as e:
        print(f"❌ Test échoué: {e}")

In [ ]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

def check_sections_details(path):
    path = Path(path)

    print(f"\n📂 Vérification de la FAISS store : {path}")

    # Chargement SAFE (important)
    store = FAISS.load_local(
        folder_path=path,
        embeddings=None,
        allow_dangerous_deserialization=True
    )

    docs = list(store.docstore._dict.values())

    print(f"📄 Nombre de documents trouvés : {len(docs)}\n")

    missing = 0
    for i, d in enumerate(docs):
        meta = d.metadata or {}
        if "sections_details" not in meta:
            print(f"❌ Doc {i} : sections_details manquant")
            missing += 1
        else:
            print(f"✔️ Doc {i} : sections_details présent")

    print("\n==============================")
    if missing == 0:
        print("✅ Tous les documents contiennent sections_details !")
    else:
        print(f"⚠️ {missing} document(s) n'ont PAS sections_details.")

if __name__ == "__main__":
    check_sections_details("./faiss_index")
